# Merge L3 Annotations from Sub-Objects into Main BBKNN All-Cells Object

## Overview

**Purpose**: Transfer refined L3 (and L2) cell type annotations from 5 lineage-specific sub-analyses into the main BBKNN-integrated all-cells object.

**Workflow**:
1. Load MAIN all-cells object (backed mode for memory efficiency)
2. Load SUB-objects metadata (epithelial, myeloid, T/NK, stromal, B cell)
3. Filter MAIN: remove cells present in MAIN but absent from ALL sub-objects
4. Transfer L3 annotations with robust validation:
   - If L3 is pure numeric (e.g., "0", "12") → convert to `{L2}_c{L3}`
   - If L3 is single character (e.g., "A") → keep as is or combine with L2
5. Save updated MAIN + mapping tables + QC reports
6. Generate dotplot visualization by L3 and L2

**Key Features**:
- Memory-optimized: backed mode loading + subset before to_memory
- Robust L3 normalization: handles numeric/character edge cases
- Provenance tracking: records source sub-object for each L3 assignment
- Comprehensive QC: overlap reports, crosstabs, missing marker lists

## 0. Environment Setup & Configuration

In [1]:
import os
import re
import gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Limit BLAS threads to prevent oversubscription on HPC
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
sc.settings.n_jobs = 24

NOW = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Run timestamp: {NOW}")

Run timestamp: 20260209_205626


In [2]:
# =======================
# INPUT PATHS
# =======================

# MAIN all-cells BBKNN-integrated object
MAIN_H5AD = "/home/h2048/data/py/1128/bbknn_annotation_analysis/adata_bbknn_annotated_corrected_filtered.h5ad"

# SUB-objects with refined L3 annotations (all 5 lineages)
SUB_H5ADS = [
    "/home/h2048/data/py/0209/epithelial_viz_L3_v1_0/epithelial_with_L3_annotations.h5ad",
    "/home/h2048/data/py/0128/myeloid_analysis_unified/results/subcluster_unified_v2_20260128/adata_myeloid_subclustered_FINAL_v2_20260128.h5ad",
    "/home/h2048/data/py/0129/tnk_analysis_unified/results/subcluster_unified_v2_20260129/adata_tnk_subclustered_FINAL_v2_0_1_20260129.h5ad",
    "/home/h2048/data/py/0120/stromal_analysis_unified/results/subcluster_unified_v2_20260128/adata_stromal_subclustered_FINAL_v2_20260128.h5ad",
    "/home/h2048/data/py/0119/bcell_analysis/results/subcluster_v2_20260119/adata_bcell_subclustered_FINAL_v2_20260119.h5ad",
]

# =======================
# COLUMN NAME DETECTION
# =======================

# Candidates for auto-detection (priority order)
L2_CANDIDATES = [
    "cell_type_L2", "celltype_L2", "CellType_L2", "L2",
    "cell_type", "celltype", "major_cell_type",
]
L3_CANDIDATES = [
    "cell_type_L3", "celltype_L3", "CellType_L3", "L3",
    "subcluster", "subcluster_id", "leiden", "leiden_sub",
]

# Target column names in MAIN
MAIN_L2_KEY = "cell_type_L2"
MAIN_L3_KEY = "cell_type_L3"

# =======================
# OUTPUT PATHS
# =======================

OUTDIR = Path(MAIN_H5AD).parent / f"merge_L3_from_subobjects_{NOW}"
FIGDIR = OUTDIR / "figures"
TABDIR = OUTDIR / "tables"

OUTDIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR.mkdir(parents=True, exist_ok=True)

print(f"\n[CONFIG] MAIN object: {MAIN_H5AD}")
print(f"[CONFIG] Output directory: {OUTDIR}")
print(f"[CONFIG] Number of sub-objects: {len(SUB_H5ADS)}")


[CONFIG] MAIN object: /home/h2048/data/py/1128/bbknn_annotation_analysis/adata_bbknn_annotated_corrected_filtered.h5ad
[CONFIG] Output directory: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626
[CONFIG] Number of sub-objects: 5


## 1. Helper Functions

In [9]:
def pick_first_existing(cols, candidates):
    """Return first candidate column name that exists in cols, else None."""
    for c in candidates:
        if c in cols:
            return c
    return None


def is_numeric_only(x: str) -> bool:
    """Check if string is purely numeric (e.g., '0', '123')."""
    return bool(re.fullmatch(r"\d+", x))


def is_single_char(x: str) -> bool:
    """Check if string is single character."""
    return len(x) == 1


def normalize_str(x):
    """Convert to clean string, return None for NA/empty values."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "" or s.lower() in {"na", "nan", "none", "null"}:
        return None
    return s


def fix_l3_with_l2(l2, l3):
    """
    Normalize L3 annotation:
    - If L3 is already 'c0', 'c1', etc. → combine with L2 if available
    - If L3 is pure numeric (e.g., '0', '12') → convert to '{L2}_c{L3}'
    - If L3 is single character (e.g., 'A') → keep as is or combine with L2
    - Otherwise → keep original L3
    
    Args:
        l2: L2 cell type string
        l3: L3 cell type string
    
    Returns:
        Normalized L3 string or None
    """
    l2s = normalize_str(l2)
    l3s = normalize_str(l3)
    
    if l3s is None:
        return None

    # Handle existing 'c0' format
    if re.fullmatch(r"c\d+", l3s):
        return f"{l2s}_{l3s}" if l2s else l3s

    # Pure numeric or single character → add prefix
    if is_numeric_only(l3s) or is_single_char(l3s):
        if l2s:
            return f"{l2s}_c{l3s}"
        else:
            # Fallback: add 'c' prefix for numeric
            return f"c{l3s}" if is_numeric_only(l3s) else l3s

    return l3s


def load_sub_obs_backed(h5ad_path: str):
    """
    Load sub-object in backed mode (read-only) to minimize memory.
    Automatically removes cell type prefixes from obs_names (e.g., "celltype::barcode" → "barcode")
    
    Returns:
        obs_min: DataFrame with obs_names as index and L2/L3 columns
        l2_key: Detected L2 column name (or None)
        l3_key: Detected L3 column name (or None)
    """
    ad = sc.read_h5ad(h5ad_path, backed="r")
    cols = list(ad.obs.columns)

    l2_key = pick_first_existing(cols, L2_CANDIDATES)
    l3_key = pick_first_existing(cols, L3_CANDIDATES)

    # Extract minimal obs subset
    use_cols = []
    if l2_key:
        use_cols.append(l2_key)
    if l3_key and l3_key not in use_cols:
        use_cols.append(l3_key)

    obs_min = ad.obs[use_cols].copy() if use_cols else pd.DataFrame(index=ad.obs_names.copy())
    obs_min.index = ad.obs_names.copy()

    # ========== CRITICAL: Remove cell type prefixes ==========
    # Some sub-objects have obs_names like "celltype::barcode"
    # Extract only the barcode part (after "::")
    if any("::" in str(idx) for idx in obs_min.index[:100]):  # Check first 100 cells
        print(f"      ⚠️  Detected '::' prefix in cell IDs, removing...")
        original_count = len(obs_min)
        
        # Split and keep only the part after "::"
        obs_min.index = obs_min.index.map(lambda x: x.split("::")[-1] if "::" in str(x) else x)
        
        # Remove duplicates if any (shouldn't happen but be safe)
        obs_min = obs_min[~obs_min.index.duplicated(keep='first')]
        
        if len(obs_min) < original_count:
            print(f"      ⚠️  Removed {original_count - len(obs_min)} duplicate cells after prefix removal")
        
        print(f"      ✓ Cleaned cell IDs: {len(obs_min):,} cells")
    # ==========================================================

    # Close file handle
    ad.file.close()

    return obs_min, l2_key, l3_key


print("[INFO] Helper functions loaded")

[INFO] Helper functions loaded


## 2. Load MAIN Object (Backed Mode)

In [10]:
# Load MAIN in backed mode first (to get obs_names without loading expression matrix)
print("\n[STEP] Loading MAIN object in backed mode...")
main_backed = sc.read_h5ad(MAIN_H5AD, backed="r")
main_cells = set(main_backed.obs_names)

print(f"  MAIN shape (backed): {main_backed.n_obs:,} cells × {main_backed.n_vars:,} genes")
print(f"  MAIN obs columns (first 20): {list(main_backed.obs.columns)[:20]}")
print(f"  Total MAIN cells: {len(main_cells):,}")


[STEP] Loading MAIN object in backed mode...
  MAIN shape (backed): 418,161 cells × 58,184 genes
  MAIN obs columns (first 20): ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'plateID', 'status', 'donorID', 'cDate', 'age', 'sex', 'cellType', 'percent.mt', 'percent.ribo', 'tissue', 'tissue_sampling_method', 'dataset', 'sample', 'percent.rb', 'decontX_contamination', 'decontX_clusters', 'nCount_decontXcounts']
  Total MAIN cells: 418,161


## 3. Load SUB Objects Metadata (Backed Mode)

In [11]:
# Load sub-objects' obs metadata only (backed mode)
sub_infos = []
all_sub_cells_union = set()

print("\n[STEP] Loading SUB-objects metadata (backed='r'):")
for i, p in enumerate(SUB_H5ADS, 1):
    p = str(p)
    fname = Path(p).name
    print(f"\n  [{i}/{len(SUB_H5ADS)}] {fname}")
    
    obs_min, l2_key, l3_key = load_sub_obs_backed(p)
    
    print(f"      Cells: {obs_min.shape[0]:,}")
    print(f"      Detected L2: {l2_key}")
    print(f"      Detected L3: {l3_key}")
    
    sub_infos.append({
        "path": p,
        "obs": obs_min,
        "l2_key": l2_key,
        "l3_key": l3_key,
    })
    all_sub_cells_union.update(obs_min.index)

print(f"\n[INFO] Union of all SUB cells: {len(all_sub_cells_union):,}")


[STEP] Loading SUB-objects metadata (backed='r'):

  [1/5] epithelial_with_L3_annotations.h5ad
      Cells: 278,553
      Detected L2: cell_type
      Detected L3: cell_type_L3

  [2/5] adata_myeloid_subclustered_FINAL_v2_20260128.h5ad
      ⚠️  Detected '::' prefix in cell IDs, removing...
      ✓ Cleaned cell IDs: 54,743 cells
      Cells: 54,743
      Detected L2: cell_type_L2
      Detected L3: cell_type_L3

  [3/5] adata_tnk_subclustered_FINAL_v2_0_1_20260129.h5ad
      ⚠️  Detected '::' prefix in cell IDs, removing...
      ✓ Cleaned cell IDs: 44,833 cells
      Cells: 44,833
      Detected L2: cell_type_L2
      Detected L3: cell_type_L3

  [4/5] adata_stromal_subclustered_FINAL_v2_20260128.h5ad
      ⚠️  Detected '::' prefix in cell IDs, removing...
      ✓ Cleaned cell IDs: 60,828 cells
      Cells: 60,828
      Detected L2: cell_type_L2
      Detected L3: cell_type_L3

  [5/5] adata_bcell_subclustered_FINAL_v2_20260119.h5ad
      Cells: 11,570
      Detected L2: cell_type_L2

In [12]:
# ================================================================================
# INSPECTION CELL: Check Cell ID Formats Across MAIN and SUB Objects
# ================================================================================

print("="*80)
print("CELL ID FORMAT INSPECTION")
print("="*80)

# ---- 1) MAIN object cell ID examples ----
print("\n[MAIN OBJECT]")
print(f"  Total cells: {len(main_cells):,}")
print(f"  Cell ID examples (first 10):")
main_cells_list = sorted(list(main_cells))[:10]
for idx, cell_id in enumerate(main_cells_list, 1):
    print(f"    {idx}. {cell_id}")

# Check ID format pattern
print(f"\n  Cell ID format analysis:")
print(f"    - Min length: {min(len(c) for c in main_cells)}")
print(f"    - Max length: {max(len(c) for c in main_cells)}")
print(f"    - Contains '-': {any('-' in c for c in list(main_cells)[:100])}")
print(f"    - Contains '_': {any('_' in c for c in list(main_cells)[:100])}")

# ---- 2) SUB objects cell ID examples ----
print("\n" + "="*80)
print("SUB-OBJECTS CELL IDs")
print("="*80)

for i, info in enumerate(sub_infos, 1):
    fname = Path(info["path"]).name
    obs = info["obs"]
    
    print(f"\n[{i}/{len(sub_infos)}] {fname}")
    print(f"  Total cells: {obs.shape[0]:,}")
    print(f"  Cell ID examples (first 5):")
    
    cell_ids = obs.index.tolist()[:5]
    for idx, cell_id in enumerate(cell_ids, 1):
        print(f"    {idx}. {cell_id}")
    
    # Check format
    all_ids = obs.index.tolist()
    print(f"  Format analysis:")
    print(f"    - Min length: {min(len(c) for c in all_ids)}")
    print(f"    - Max length: {max(len(c) for c in all_ids)}")
    print(f"    - Contains '-': {any('-' in c for c in all_ids[:100])}")
    print(f"    - Contains '_': {any('_' in c for c in all_ids[:100])}")
    
    # Check overlap with MAIN
    overlap = set(all_ids).intersection(main_cells)
    print(f"  Overlap with MAIN: {len(overlap):,} / {len(all_ids):,} ({100*len(overlap)/len(all_ids):.2f}%)")
    
    if len(overlap) == 0:
        print(f"  ⚠️  WARNING: No overlap with MAIN - check cell ID formatting!")
        # Show MAIN vs SUB ID comparison
        print(f"\n  Comparison example:")
        print(f"    MAIN ID:  {main_cells_list[0]}")
        print(f"    SUB ID:   {all_ids[0]}")

# ---- 3) Cross-check all SUBs vs MAIN ----
print("\n" + "="*80)
print("CROSS-CHECK: SUB UNION vs MAIN")
print("="*80)

print(f"\nMAIN cells: {len(main_cells):,}")
print(f"SUB union cells: {len(all_sub_cells_union):,}")
print(f"Intersection: {len(main_cells.intersection(all_sub_cells_union)):,}")
print(f"MAIN only: {len(main_cells - all_sub_cells_union):,}")
print(f"SUB only: {len(all_sub_cells_union - main_cells):,}")

# If no overlap, show diagnostic
if len(main_cells.intersection(all_sub_cells_union)) == 0:
    print("\n❌ CRITICAL: No cells in common between MAIN and any SUB!")
    print("\nPossible causes:")
    print("  1. Cell ID format mismatch (e.g., barcodes vs modified IDs)")
    print("  2. Different preprocessing pipelines applied")
    print("  3. MAIN object is not the parent of these SUB objects")
    print("\nAction needed:")
    print("  - Verify MAIN_H5AD path is correct")
    print("  - Check if SUB objects have modified cell IDs (e.g., added prefixes/suffixes)")
    print("  - Ensure SUB objects were derived from this MAIN object")
elif len(main_cells.intersection(all_sub_cells_union)) < 0.5 * len(main_cells):
    print("\n⚠️  WARNING: Low overlap - only {:.1f}% of MAIN cells found in SUBs".format(
        100 * len(main_cells.intersection(all_sub_cells_union)) / len(main_cells)
    ))

# ---- 4) Sample ID matching examples ----
print("\n" + "="*80)
print("SAMPLE ID MATCHING TEST")
print("="*80)

# Take first 5 cells from MAIN, check which SUBs contain them
test_cells = main_cells_list[:5]
print("\nChecking 5 random MAIN cells across all SUBs:")
for cell in test_cells:
    print(f"\n  Cell: {cell}")
    found_in = []
    for info in sub_infos:
        fname = Path(info["path"]).stem
        if cell in info["obs"].index:
            found_in.append(fname)
    
    if found_in:
        print(f"    ✓ Found in: {', '.join(found_in)}")
    else:
        print(f"    ✗ Not found in any SUB object")

print("\n" + "="*80)
print("INSPECTION COMPLETE")
print("="*80)

CELL ID FORMAT INSPECTION

[MAIN OBJECT]
  Total cells: 418,161
  Cell ID examples (first 10):
    1. merged_seurat_Jennifer_P_Wang_2021_HC0301_AAACTATCAGTC
    2. merged_seurat_Jennifer_P_Wang_2021_HC0301_AACACTTATAGC
    3. merged_seurat_Jennifer_P_Wang_2021_HC0301_AACTTGGATTCA
    4. merged_seurat_Jennifer_P_Wang_2021_HC0301_AATTTCCTCGCG
    5. merged_seurat_Jennifer_P_Wang_2021_HC0301_ACTCAGTTCCTG
    6. merged_seurat_Jennifer_P_Wang_2021_HC0301_AGGAAAGGCCCG
    7. merged_seurat_Jennifer_P_Wang_2021_HC0301_AGGCGCAACCGG
    8. merged_seurat_Jennifer_P_Wang_2021_HC0301_ATATAAACGGTA
    9. merged_seurat_Jennifer_P_Wang_2021_HC0301_ATCCGAGAACGA
    10. merged_seurat_Jennifer_P_Wang_2021_HC0301_CACAACACCGTC

  Cell ID format analysis:
    - Min length: 54
    - Max length: 117
    - Contains '-': True
    - Contains '_': True

SUB-OBJECTS CELL IDs

[1/5] epithelial_with_L3_annotations.h5ad
  Total cells: 278,553
  Cell ID examples (first 5):
    1. merged_seurat_Jennifer_P_Wang_2021_HC4

## 4. Filter MAIN: Keep Only Cells in SUB Union

In [13]:
# Compute cells to keep (intersection of MAIN and SUB union)
keep_cells = main_cells.intersection(all_sub_cells_union)
drop_cells = main_cells - keep_cells

print("\n[STEP] Filtering MAIN by SUB union (obs_names match):")
print(f"  MAIN cells:         {len(main_cells):,}")
print(f"  Cells to KEEP:      {len(keep_cells):,}")
print(f"  Cells to DROP:      {len(drop_cells):,}")
print(f"  Retention rate:     {100*len(keep_cells)/len(main_cells):.2f}%")

# Export dropped cell list
if len(drop_cells) > 0:
    dropped_df = pd.Series(sorted(drop_cells), name="dropped_cells")
    dropped_df.to_csv(TABDIR / "main_cells_dropped_not_in_any_sub.csv", index=False)
    print(f"  ✓ Dropped cells list saved: {TABDIR / 'main_cells_dropped_not_in_any_sub.csv'}")


[STEP] Filtering MAIN by SUB union (obs_names match):
  MAIN cells:         418,161
  Cells to KEEP:      412,351
  Cells to DROP:      5,810
  Retention rate:     98.61%
  ✓ Dropped cells list saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/tables/main_cells_dropped_not_in_any_sub.csv


In [14]:
# Subset MAIN to keep_cells and load into memory
print("\n[STEP] Loading filtered MAIN subset into memory...")
keep_cells_list = sorted(keep_cells)  # Convert set to sorted list for indexing

main = main_backed[keep_cells_list, :].to_memory()
main_backed.file.close()  # Close backed file handle
del main_backed
gc.collect()

print(f"  MAIN (in-memory subset): {main.n_obs:,} cells × {main.n_vars:,} genes")
print(f"  Memory usage: ~{main.X.data.nbytes / 1e9:.2f} GB (expression matrix only)")


[STEP] Loading filtered MAIN subset into memory...
  MAIN (in-memory subset): 412,351 cells × 58,184 genes
  Memory usage: ~6.06 GB (expression matrix only)


## 5. Transfer L3 Annotations from SUB to MAIN

In [15]:
# Initialize target columns in MAIN
if MAIN_L2_KEY not in main.obs.columns:
    main.obs[MAIN_L2_KEY] = pd.NA
if MAIN_L3_KEY not in main.obs.columns:
    main.obs[MAIN_L3_KEY] = pd.NA

# Track provenance (which sub-object provided the L3)
SRC_KEY = f"{MAIN_L3_KEY}_source"
if SRC_KEY not in main.obs.columns:
    main.obs[SRC_KEY] = pd.NA

print(f"\n[INFO] Target columns in MAIN:")
print(f"  L2: {MAIN_L2_KEY}")
print(f"  L3: {MAIN_L3_KEY}")
print(f"  Source: {SRC_KEY}")


[INFO] Target columns in MAIN:
  L2: cell_type_L2
  L3: cell_type_L3
  Source: cell_type_L3_source


In [16]:
# Transfer L3 from each sub-object (later subs override earlier ones)
main_index = main.obs_names
main_l2_series = main.obs[MAIN_L2_KEY].astype("object").copy()
main_l3_series = main.obs[MAIN_L3_KEY].astype("object").copy()
src_series = main.obs[SRC_KEY].astype("object").copy()

transfer_rows = []  # For mapping table

print("\n[STEP] Transferring L3 from SUB-objects (in order given):")
for i, info in enumerate(sub_infos, 1):
    obs = info["obs"]
    l2k = info["l2_key"]
    l3k = info["l3_key"]
    src = Path(info["path"]).name

    # Only cells that exist in MAIN
    overlap = obs.index.intersection(main_index)
    print(f"\n  [{i}/{len(sub_infos)}] {src}")
    print(f"      Overlap with MAIN: {len(overlap):,}")

    if len(overlap) == 0:
        print(f"      ⚠️  No overlap, skipping")
        continue
    
    if l3k is None:
        print(f"      ⚠️  No L3 column detected, skipping")
        continue

    # Extract L2 and L3 values for overlap cells
    l2_vals = obs.loc[overlap, l2k] if l2k else pd.Series(index=overlap, data=[None]*len(overlap))
    l3_vals = obs.loc[overlap, l3k]

    # Fix L3 and update MAIN
    l3_fixed_list = []
    for cell, l2v, l3v in zip(overlap, l2_vals.values, l3_vals.values):
        l2s = normalize_str(l2v)
        l3s = normalize_str(l3v)
        l3f = fix_l3_with_l2(l2s, l3s)

        # Update L2 if MAIN's L2 is missing
        if l2s is not None:
            current_l2 = main_l2_series.loc[cell]
            if pd.isna(current_l2) or normalize_str(current_l2) is None:
                main_l2_series.loc[cell] = l2s

        # Update L3 and source
        if l3f is not None:
            main_l3_series.loc[cell] = l3f
            src_series.loc[cell] = src

        l3_fixed_list.append(l3f)

    # Record mapping table
    transfer_rows.append(pd.DataFrame({
        "cell": overlap,
        "L2_from_sub": [normalize_str(x) for x in l2_vals.values],
        "L3_raw_from_sub": [normalize_str(x) for x in l3_vals.values],
        "L3_fixed": l3_fixed_list,
        "source": src,
    }))
    
    print(f"      ✓ Transferred {sum(x is not None for x in l3_fixed_list)} L3 annotations")

print("\n[INFO] L3 transfer complete")


[STEP] Transferring L3 from SUB-objects (in order given):

  [1/5] epithelial_with_L3_annotations.h5ad
      Overlap with MAIN: 263,203
      ✓ Transferred 263203 L3 annotations

  [2/5] adata_myeloid_subclustered_FINAL_v2_20260128.h5ad
      Overlap with MAIN: 54,552
      ✓ Transferred 54552 L3 annotations

  [3/5] adata_tnk_subclustered_FINAL_v2_0_1_20260129.h5ad
      Overlap with MAIN: 44,698
      ✓ Transferred 44698 L3 annotations

  [4/5] adata_stromal_subclustered_FINAL_v2_20260128.h5ad
      Overlap with MAIN: 38,337
      ✓ Transferred 38337 L3 annotations

  [5/5] adata_bcell_subclustered_FINAL_v2_20260119.h5ad
      Overlap with MAIN: 11,561
      ✓ Transferred 11561 L3 annotations

[INFO] L3 transfer complete


In [17]:
# Write back to MAIN AnnData
main.obs[MAIN_L2_KEY] = pd.Series(main_l2_series, index=main_index).astype("category")
main.obs[MAIN_L3_KEY] = pd.Series(main_l3_series, index=main_index).astype("category")
main.obs[SRC_KEY] = pd.Series(src_series, index=main_index).astype("category")

print("\n[QC] MAIN L2 value counts (top 30):")
print(main.obs[MAIN_L2_KEY].value_counts().head(30))

print("\n[QC] MAIN L3 value counts (top 30):")
print(main.obs[MAIN_L3_KEY].value_counts().head(30))

print("\n[QC] L3 source distribution:")
print(main.obs[SRC_KEY].value_counts())


[QC] MAIN L2 value counts (top 30):
cell_type_L2
Epithelial                                263203
Alveolar macrophages                       24146
Macrophages                                13895
Fibro_adventitial                          11223
CD16+ NK cells                             10963
Trm cytotoxic T cells                       9443
Classical monocytes                         8895
Endothelia_vascular_Cap_g                   6067
CD8+ Trm cytotoxic T cells                  5334
Endothelia_vascular_venous_systemic         5144
Tem/Effector helper T cells                 3845
Tem/Trm cytotoxic T cells                   3652
Mast cells                                  3630
Fibro_peribronchial                         3500
Naive B cells                               3012
NK cells                                    2847
Memory B cells                              2509
Germinal center B cells                     2246
CD8+ Tem/Trm cytotoxic T cells              2123
Plasma cells       

## 6. Export Mapping Tables and QC Reports

In [18]:
# Export L3 transfer mapping table
if transfer_rows:
    transfer_df = pd.concat(transfer_rows, axis=0, ignore_index=True)
else:
    transfer_df = pd.DataFrame(columns=["cell", "L2_from_sub", "L3_raw_from_sub", "L3_fixed", "source"])

mapping_file = TABDIR / "l3_transfer_mapping_table.csv"
transfer_df.to_csv(mapping_file, index=False)
print(f"\n[EXPORT] L3 transfer mapping table: {mapping_file}")
print(f"  Total transfers: {len(transfer_df):,}")


[EXPORT] L3 transfer mapping table: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/tables/l3_transfer_mapping_table.csv
  Total transfers: 412,351


In [19]:
# Export L2 vs L3 crosstab
crosstab = pd.crosstab(main.obs[MAIN_L2_KEY], main.obs[MAIN_L3_KEY])
crosstab_file = TABDIR / "L2_vs_L3_crosstab.csv"
crosstab.to_csv(crosstab_file)
print(f"\n[EXPORT] L2 vs L3 crosstab: {crosstab_file}")
print(f"  L2 categories: {crosstab.shape[0]}")
print(f"  L3 categories: {crosstab.shape[1]}")


[EXPORT] L2 vs L3 crosstab: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/tables/L2_vs_L3_crosstab.csv
  L2 categories: 55
  L3 categories: 152


In [20]:
# Export per-sub overlap report
overlap_report = []
for info in sub_infos:
    sub_cells = set(info["obs"].index)
    overlap_n = len(set(main.obs_names).intersection(sub_cells))
    overlap_report.append({
        "sub_file": Path(info["path"]).name,
        "sub_cells": len(sub_cells),
        "overlap_with_final_main": overlap_n,
        "overlap_rate": f"{100*overlap_n/len(sub_cells):.2f}%" if len(sub_cells) > 0 else "N/A",
    })

overlap_df = pd.DataFrame(overlap_report).sort_values("overlap_with_final_main", ascending=False)
overlap_file = TABDIR / "overlap_report.csv"
overlap_df.to_csv(overlap_file, index=False)

print(f"\n[EXPORT] Overlap report: {overlap_file}")
print(overlap_df)


[EXPORT] Overlap report: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/tables/overlap_report.csv
                                            sub_file  sub_cells  \
0                epithelial_with_L3_annotations.h5ad     278553   
1  adata_myeloid_subclustered_FINAL_v2_20260128.h5ad      54743   
2  adata_tnk_subclustered_FINAL_v2_0_1_20260129.h5ad      44833   
3  adata_stromal_subclustered_FINAL_v2_20260128.h5ad      60828   
4    adata_bcell_subclustered_FINAL_v2_20260119.h5ad      11570   

   overlap_with_final_main overlap_rate  
0                   263203       94.49%  
1                    54552       99.65%  
2                    44698       99.70%  
3                    38337       63.03%  
4                    11561       99.92%  


## 7. Save Updated MAIN AnnData

In [21]:
# Save updated MAIN with L3 annotations
UPDATED_H5AD = OUTDIR / f"{Path(MAIN_H5AD).stem}_L3_merged_{NOW}.h5ad"

print(f"\n[STEP] Writing updated MAIN h5ad...")
print(f"  Output: {UPDATED_H5AD}")
print(f"  Shape: {main.n_obs:,} cells × {main.n_vars:,} genes")

main.write(UPDATED_H5AD, compression="gzip")
print(f"  ✓ Saved successfully")
print(f"  File size: {UPDATED_H5AD.stat().st_size / 1e9:.2f} GB")


[STEP] Writing updated MAIN h5ad...
  Output: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/adata_bbknn_annotated_corrected_filtered_L3_merged_20260209_205626.h5ad
  Shape: 412,351 cells × 58,184 genes
  ✓ Saved successfully
  File size: 9.68 GB


## 8. Generate DotPlot Visualization

In [22]:
# Define multi-lineage marker panel
MARKERS = {
    "Epithelial": ["EPCAM", "KRT8", "KRT18", "KRT5", "FOXJ1", "SCGB1A1", "MUC5AC", "MUC5B", "AGER", "SFTPC"],
    "T_NK": ["CD3D", "CD3E", "TRAC", "CD4", "IL7R", "CD8A", "CD8B", "NKG7", "GNLY", "KLRD1", "FCGR3A"],
    "B": ["MS4A1", "CD79A", "CD74", "HLA-DRA", "MZB1", "XBP1", "TNFRSF17", "JCHAIN"],
    "Myeloid": ["LYZ", "S100A8", "S100A9", "FCER1G", "LST1", "TYROBP", "APOE", "C1QA", "C1QB", "C1QC"],
    "Stromal": ["COL1A1", "COL1A2", "DCN", "LUM", "COL3A1", "PDGFRA", "ACTA2", "TAGLN", "MYH11"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "EMCN", "RAMP2", "PLVAP", "CLDN5"],
    "Proliferation": ["MKI67", "TOP2A", "STMN1", "TK1"],
}

# Flatten and keep only genes present in data
all_markers = []
for lineage, genes in MARKERS.items():
    all_markers.extend(genes)
all_markers = list(dict.fromkeys(all_markers))  # Unique, preserve order

present_markers = [g for g in all_markers if g in main.var_names]
missing_markers = [g for g in all_markers if g not in main.var_names]

print(f"\n[INFO] Marker genes for dotplot:")
print(f"  Total markers: {len(all_markers)}")
print(f"  Present: {len(present_markers)}")
print(f"  Missing: {len(missing_markers)}")

if len(missing_markers) > 0:
    missing_file = TABDIR / "dotplot_missing_markers.csv"
    pd.Series(missing_markers, name="missing_markers").to_csv(missing_file, index=False)
    print(f"  ⚠️  Missing markers saved: {missing_file}")


[INFO] Marker genes for dotplot:
  Total markers: 59
  Present: 59
  Missing: 0


In [23]:
# Generate dotplot by L3
use_raw = main.raw is not None
print(f"\n[STEP] Generating dotplot by {MAIN_L3_KEY}...")
print(f"  use_raw: {use_raw}")

sc.settings.figdir = str(FIGDIR)

try:
    fig = sc.pl.dotplot(
        main,
        var_names=present_markers,
        groupby=MAIN_L3_KEY,
        use_raw=use_raw,
        standard_scale="var",
        show=False,
        return_fig=True
    )
    
    dotplot_l3_file = FIGDIR / f"dotplot_markers_by_{MAIN_L3_KEY}.pdf"
    fig.savefig(dotplot_l3_file, bbox_inches="tight")
    plt.close("all")
    print(f"  ✓ Saved: {dotplot_l3_file}")
except Exception as e:
    print(f"  ⚠️  Dotplot by L3 failed: {e}")


[STEP] Generating dotplot by cell_type_L3...
  use_raw: False


/home/h2048/miniconda3/envs/bbknn_env/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot_ax.scatter(x, y, **kwds)


  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/dotplot_markers_by_cell_type_L3.pdf


In [24]:
# Generate dotplot by L2
print(f"\n[STEP] Generating dotplot by {MAIN_L2_KEY}...")

try:
    fig2 = sc.pl.dotplot(
        main,
        var_names=present_markers,
        groupby=MAIN_L2_KEY,
        use_raw=use_raw,
        standard_scale="var",
        show=False,
        return_fig=True
    )
    
    dotplot_l2_file = FIGDIR / f"dotplot_markers_by_{MAIN_L2_KEY}.pdf"
    fig2.savefig(dotplot_l2_file, bbox_inches="tight")
    plt.close("all")
    print(f"  ✓ Saved: {dotplot_l2_file}")
except Exception as e:
    print(f"  ⚠️  Dotplot by L2 failed: {e}")


[STEP] Generating dotplot by cell_type_L2...


/home/h2048/miniconda3/envs/bbknn_env/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot_ax.scatter(x, y, **kwds)


  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/dotplot_markers_by_cell_type_L2.pdf


## 9. Mechanosensing Gene Expression Analysis

### Background

Mechanotransduction is critical in respiratory epithelial biology and disease. This section visualizes expression patterns of 15 functional categories of mechanosensing genes:

**Signal Flow Hierarchy**:
1. **Primary Sensors** (Panel 1): Ion channels detecting mechanical stimuli
2. **Surface Receptors** (Panel 2): Integrins and endothelial shear sensors
3. **Cellular Transduction** (Panel 3): Focal adhesions, adherens junctions, membrane buffers
4. **Nuclear & ECM** (Panel 4): LINC complex, YAP/TAZ pathway, ECM ligands

### 9.1 Define Mechanosensing Gene Sets

In [25]:
# Comprehensive mechanosensing gene classification (15 functional groups)
# Based on mechanistic hierarchy: sensors → receptors → transducers → transcriptional outputs

from collections import OrderedDict

MECH_GENE_SETS = OrderedDict([
    
    # ===== PANEL 1: Primary Mechanosensitive Sensors =====
    
    ("Classic mechanosensitive channels", [
        "PIEZO1", "PIEZO2"
    ]),
    
    ("TMC / MET complex", [
        "TMC1", "TMC2", "TMC3", "TMC4", "TMC5", "TMC6", "TMC7", "TMC8",
        "CDH23", "PCDH15", "LHFPL5", "TMIE", "CIB2"
    ]),
    
    ("TRP-related mechanosensors", [
        "TRPV4", "TRPC1", "TRPC6", "TRPM7", "TRPA1"
    ]),
    
    ("K2P stretch-sensitive K+ channels", [
        "KCNK2", "KCNK10", "KCNK4"
    ]),
    
    ("ENaC / ASIC family", [
        "SCNN1A", "SCNN1B", "SCNN1G",
        "ASIC1", "ASIC2", "ASIC3"
    ]),
    
    # ===== PANEL 2: Surface Receptors & Complexes =====
    
    ("Integrins (ECM force receptors)", [
        "ITGA1", "ITGA2", "ITGA3", "ITGA4", "ITGA5", "ITGA6", "ITGA7", "ITGA8", "ITGA9",
        "ITGA10", "ITGA11", "ITGAD", "ITGAE", "ITGAL", "ITGAM", "ITGAV", "ITGAX",
        "ITGB1", "ITGB2", "ITGB3", "ITGB4", "ITGB5", "ITGB6", "ITGB7", "ITGB8"
    ]),
    
    ("Endothelial shear stress complex", [
        "PECAM1", "CDH5", "KDR"
    ]),
    
    ("Stretch-associated GPCR / RTK", [
        "AGTR1", "EGFR", "KDR"  # KDR duplicate resolved by priority
    ]),
    
    # ===== PANEL 3: Cellular Mechanotransduction Machinery =====
    
    ("Focal adhesion & molecular clutch", [
        "TLN1", "TLN2", "VCL",
        "FERMT1", "FERMT2", "FERMT3",
        "PTK2", "SRC", "PXN", "ZYX", "CRK", "CRKL",
        "ILK", "LIMS1", "LIMS2", "PARVA", "PARVB", "PARVG",
        "ACTN1", "ACTN4",
        "FLNA", "FLNB",
        "VASP", "ENAH",
        "MYH9", "MYH10", "MYL9"
    ]),
    
    ("Adherens junction mechanosensors", [
        "CDH1", "CDH2",
        "CTNNB1", "JUP",
        "CTNNA1", "CTNNA2",
        "VCL"  # VCL duplicate resolved by priority (focal adhesion wins)
    ]),
    
    ("Caveolae (membrane tension buffer)", [
        "CAV1", "CAV2",
        "CAVIN1", "CAVIN2", "CAVIN3", "CAVIN4"
    ]),
    
    ("Spectrin-ankyrin network", [
        "SPTA1", "SPTB", "SPTAN1", "SPTBN1", "SPTBN2",
        "ANK1", "ANK2", "ANK3"
    ]),
    
    # ===== PANEL 4: Nuclear Mechanotransduction & ECM =====
    
    ("Nuclear mechanotransduction (LINC)", [
        "SYNE1", "SYNE2", "SYNE3", "SYNE4",
        "SUN1", "SUN2",
        "LMNA", "LMNB1", "LMNB2"
    ]),
    
    ("Mechano-transcriptional output", [
        "YAP1", "WWTR1",
        "TEAD1", "TEAD2", "TEAD3", "TEAD4",
        "MRTFA", "MRTFB",
        "SRF"
    ]),
    
    ("ECM ligands (force coupling)", [
        "FN1",
        "COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2",
        "LAMA1", "LAMA2", "LAMA3", "LAMB1", "LAMB2", "LAMC1"
    ])
])

# Priority order for duplicate gene resolution
MECH_PRIORITY = list(MECH_GENE_SETS.keys())

print("[INFO] Mechanosensing gene sets defined")
print(f"  Total gene groups: {len(MECH_GENE_SETS)}")
total_genes_raw = sum(len(genes) for genes in MECH_GENE_SETS.values())
print(f"  Total genes (with duplicates): {total_genes_raw}")

[INFO] Mechanosensing gene sets defined
  Total gene groups: 15
  Total genes (with duplicates): 138


### 9.2 Resolve Duplicate Genes via Priority

In [26]:
def resolve_gene_duplicates(gene_sets, priority_order):
    """
    Assign each gene to ONE group based on priority order.
    Earlier groups in priority_order win for duplicated genes.
    
    Returns:
        gene_to_group: dict {gene_symbol: group_name}
        unique_genes: list of unique genes in priority order
    """
    used_genes = set()
    gene_to_group = {}
    unique_genes = []
    
    for group in priority_order:
        genes = gene_sets[group]
        for gene in genes:
            if gene not in used_genes:
                gene_to_group[gene] = group
                unique_genes.append(gene)
                used_genes.add(gene)
    
    return gene_to_group, unique_genes


# Resolve duplicates
gene_to_group, unique_genes = resolve_gene_duplicates(MECH_GENE_SETS, MECH_PRIORITY)

print(f"\n[INFO] Gene duplicate resolution:")
print(f"  Unique genes after deduplication: {len(unique_genes)}")
print(f"  Duplicates removed: {total_genes_raw - len(unique_genes)}")

# Find duplicate examples
all_genes_flat = [g for genes in MECH_GENE_SETS.values() for g in genes]
duplicates = [g for g in set(all_genes_flat) if all_genes_flat.count(g) > 1]

if duplicates:
    print(f"\n[INFO] Example duplicated genes (priority resolution applied):")
    for dup in sorted(duplicates)[:5]:
        groups_with_dup = [k for k, v in MECH_GENE_SETS.items() if dup in v]
        assigned = gene_to_group[dup]
        print(f"  {dup}: appears in {groups_with_dup}")
        print(f"    → Assigned to: {assigned}")


[INFO] Gene duplicate resolution:
  Unique genes after deduplication: 136
  Duplicates removed: 2

[INFO] Example duplicated genes (priority resolution applied):
  KDR: appears in ['Endothelial shear stress complex', 'Stretch-associated GPCR / RTK']
    → Assigned to: Endothelial shear stress complex
  VCL: appears in ['Focal adhesion & molecular clutch', 'Adherens junction mechanosensors']
    → Assigned to: Focal adhesion & molecular clutch


### 9.3 Filter Genes Present in Dataset

In [27]:
# Check which genes are present in the dataset
present_mech_genes = [g for g in unique_genes if g in main.var_names]
missing_mech_genes = [g for g in unique_genes if g not in main.var_names]

print(f"\n[INFO] Mechanosensing genes in dataset:")
print(f"  Total unique genes: {len(unique_genes)}")
print(f"  Present in dataset: {len(present_mech_genes)} ({100*len(present_mech_genes)/len(unique_genes):.1f}%)")
print(f"  Missing from dataset: {len(missing_mech_genes)}")

# Export missing genes
if missing_mech_genes:
    missing_mech_file = TABDIR / "mechanosensing_missing_genes.csv"
    missing_df = pd.DataFrame({
        "gene": missing_mech_genes,
        "group": [gene_to_group[g] for g in missing_mech_genes]
    })
    missing_df.to_csv(missing_mech_file, index=False)
    print(f"  ⚠️  Missing genes saved: {missing_mech_file}")
    print(f"\n  Top missing genes: {', '.join(missing_mech_genes[:10])}")

# Count genes per group (present only)
group_counts = {}
for gene in present_mech_genes:
    group = gene_to_group[gene]
    group_counts[group] = group_counts.get(group, 0) + 1

print(f"\n[INFO] Genes per group (present in dataset):")
for group in MECH_PRIORITY:
    count = group_counts.get(group, 0)
    print(f"  {group}: {count}")


[INFO] Mechanosensing genes in dataset:
  Total unique genes: 136
  Present in dataset: 136 (100.0%)
  Missing from dataset: 0

[INFO] Genes per group (present in dataset):
  Classic mechanosensitive channels: 2
  TMC / MET complex: 13
  TRP-related mechanosensors: 5
  K2P stretch-sensitive K+ channels: 3
  ENaC / ASIC family: 6
  Integrins (ECM force receptors): 25
  Endothelial shear stress complex: 3
  Stretch-associated GPCR / RTK: 2
  Focal adhesion & molecular clutch: 27
  Adherens junction mechanosensors: 6
  Caveolae (membrane tension buffer): 6
  Spectrin-ankyrin network: 8
  Nuclear mechanotransduction (LINC): 9
  Mechano-transcriptional output: 9
  ECM ligands (force coupling): 12


### 9.4 Prepare DotPlot Data with Group Annotations

In [28]:
# Use scanpy's dotplot to generate base data
print("\n[STEP] Generating mechanosensing dotplot data...")

use_raw = main.raw is not None
print(f"  use_raw: {use_raw}")
print(f"  Grouping by: {MAIN_L3_KEY}")

# Create dotplot object
dp = sc.pl.DotPlot(
    main,
    var_names=present_mech_genes,
    groupby=MAIN_L3_KEY,
    use_raw=use_raw,
    standard_scale='var'
)

# Extract data
dotplot_df = dp.dot_color_df.stack().reset_index()
dotplot_df.columns = ['cell_type', 'gene', 'mean_expression']

dotplot_pct = dp.dot_size_df.stack().reset_index()
dotplot_pct.columns = ['cell_type', 'gene', 'pct_expressed']

# Merge
mech_dotplot_data = dotplot_df.merge(dotplot_pct, on=['cell_type', 'gene'])

# Add group annotation
mech_dotplot_data['group'] = mech_dotplot_data['gene'].map(gene_to_group)
mech_dotplot_data['group'] = pd.Categorical(
    mech_dotplot_data['group'],
    categories=MECH_PRIORITY,
    ordered=True
)

print(f"  ✓ DotPlot data prepared: {len(mech_dotplot_data)} rows")
print(f"  Genes: {mech_dotplot_data['gene'].nunique()}")
print(f"  Cell types: {mech_dotplot_data['cell_type'].nunique()}")


[STEP] Generating mechanosensing dotplot data...
  use_raw: False
  Grouping by: cell_type_L3
  ✓ DotPlot data prepared: 20672 rows
  Genes: 136
  Cell types: 152


### 9.5 Define Panel Groupings

In [29]:
# Split into 4 mechanistic panels

# Panel 1: Primary mechanosensitive sensors (ion channels)
PANEL_1_GROUPS = [
    "Classic mechanosensitive channels",
    "TMC / MET complex",
    "TRP-related mechanosensors",
    "K2P stretch-sensitive K+ channels",
    "ENaC / ASIC family"
]

# Panel 2: Surface receptors
PANEL_2_GROUPS = [
    "Integrins (ECM force receptors)",
    "Endothelial shear stress complex",
    "Stretch-associated GPCR / RTK"
]

# Panel 3: Cellular mechanotransduction machinery
PANEL_3_GROUPS = [
    "Focal adhesion & molecular clutch",
    "Adherens junction mechanosensors",
    "Caveolae (membrane tension buffer)",
    "Spectrin-ankyrin network"
]

# Panel 4: Nuclear and ECM
PANEL_4_GROUPS = [
    "Nuclear mechanotransduction (LINC)",
    "Mechano-transcriptional output",
    "ECM ligands (force coupling)"
]

print("[INFO] Panel groupings defined:")
print(f"  Panel 1 (Primary Sensors): {len(PANEL_1_GROUPS)} groups")
print(f"  Panel 2 (Surface Receptors): {len(PANEL_2_GROUPS)} groups")
print(f"  Panel 3 (Cellular Transduction): {len(PANEL_3_GROUPS)} groups")
print(f"  Panel 4 (Nuclear & ECM): {len(PANEL_4_GROUPS)} groups")

[INFO] Panel groupings defined:
  Panel 1 (Primary Sensors): 5 groups
  Panel 2 (Surface Receptors): 3 groups
  Panel 3 (Cellular Transduction): 4 groups
  Panel 4 (Nuclear & ECM): 3 groups


### 9.6 Plot Function with Custom Colormap

In [30]:
def plot_mechanosensing_panel(data, groups, title, filename, figsize=(16, 10)):
    """
    Generate a multi-facet dotplot for mechanosensing genes.
    
    Args:
        data: DataFrame with columns [cell_type, gene, mean_expression, pct_expressed, group]
        groups: List of group names to include in this panel
        title: Panel title
        filename: Output filename
        figsize: Figure size tuple
    """
    import matplotlib.patches as mpatches
    from matplotlib.colors import LinearSegmentedColormap
    
    # Filter data for this panel
    panel_data = data[data['group'].isin(groups)].copy()
    panel_data['group'] = pd.Categorical(
        panel_data['group'],
        categories=groups,
        ordered=True
    )
    
    if len(panel_data) == 0:
        print(f"  ⚠️  No data for panel: {title}")
        return None
    
    # Custom colormap (blue to red, matching your R code)
    colors_list = [
        "#1E466E", "#376795", "#528FAD", "#72BCD5", "#AADCE0",
        "#FFE6B7", "#FFD06F", "#F7AA58", "#EF8A47", "#E76254"
    ]
    cmap = LinearSegmentedColormap.from_list("custom_cmap", colors_list, N=256)
    
    # Create figure with subplots for each group
    n_groups = len(groups)
    fig, axes = plt.subplots(1, n_groups, figsize=figsize, 
                            gridspec_kw={'wspace': 0.05})
    
    if n_groups == 1:
        axes = [axes]
    
    for idx, (ax, group) in enumerate(zip(axes, groups)):
        group_data = panel_data[panel_data['group'] == group]
        
        if len(group_data) == 0:
            ax.set_visible(False)
            continue
        
        # Pivot for plotting
        expr_pivot = group_data.pivot(index='cell_type', columns='gene', values='mean_expression')
        pct_pivot = group_data.pivot(index='cell_type', columns='gene', values='pct_expressed')
        
        # Get unique genes in original order
        genes_in_group = [g for g in present_mech_genes if g in expr_pivot.columns]
        cell_types = expr_pivot.index.tolist()
        
        # Plot dots
        for i, ct in enumerate(cell_types):
            for j, gene in enumerate(genes_in_group):
                if pd.notna(expr_pivot.loc[ct, gene]):
                    size = pct_pivot.loc[ct, gene] * 3  # Scale size
                    color = expr_pivot.loc[ct, gene]
                    
                    ax.scatter(j, i, s=size, c=[color], cmap=cmap,
                             vmin=-2, vmax=2, edgecolors='black', linewidths=0.5)
        
        # Formatting
        ax.set_xlim(-0.5, len(genes_in_group) - 0.5)
        ax.set_ylim(-0.5, len(cell_types) - 0.5)
        ax.set_xticks(range(len(genes_in_group)))
        ax.set_xticklabels(genes_in_group, rotation=45, ha='right', fontsize=9)
        
        if idx == 0:
            ax.set_yticks(range(len(cell_types)))
            ax.set_yticklabels(cell_types, fontsize=9)
        else:
            ax.set_yticks([])
        
        ax.set_title(group, fontsize=10, fontweight='bold', pad=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(False)
    
    # Overall title
    fig.suptitle(title, fontsize=14, fontweight='bold', y=0.98)
    
    # Add colorbars
    # Expression colorbar
    sm_expr = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=-2, vmax=2))
    sm_expr.set_array([])
    cbar_ax_expr = fig.add_axes([0.92, 0.55, 0.01, 0.3])
    cbar_expr = fig.colorbar(sm_expr, cax=cbar_ax_expr)
    cbar_expr.set_label('Scaled Expression', rotation=270, labelpad=15, fontsize=10)
    
    # Size legend (percentage)
    size_legend_elements = [
        mpatches.Circle((0, 0), radius=np.sqrt(25*3)/50, facecolor='gray', 
                       edgecolor='black', linewidth=0.5, label='25%'),
        mpatches.Circle((0, 0), radius=np.sqrt(50*3)/50, facecolor='gray',
                       edgecolor='black', linewidth=0.5, label='50%'),
        mpatches.Circle((0, 0), radius=np.sqrt(75*3)/50, facecolor='gray',
                       edgecolor='black', linewidth=0.5, label='75%'),
    ]
    legend = fig.legend(handles=size_legend_elements, loc='center right',
                       bbox_to_anchor=(0.99, 0.25), title='% Expressed',
                       frameon=False, fontsize=9)
    plt.setp(legend.get_title(), fontsize=10, fontweight='bold')
    
    # Save
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {filename}")
    plt.close()
    
    return fig


print("[INFO] Plot function defined")

[INFO] Plot function defined


### 9.7 Generate 4 Mechanosensing Panels

In [33]:
# Create mechanosensing-specific output directory
MECH_FIGDIR = FIGDIR / "mechanosensing"
MECH_FIGDIR.mkdir(exist_ok=True)

print("\n[STEP] Generating mechanosensing dotplots (4 panels)...\n")

# Panel 1: Primary Sensors
print("Panel 1: Primary Mechanosensitive Sensors")
plot_mechanosensing_panel(
    mech_dotplot_data,
    PANEL_1_GROUPS,
    "Mechanosensing Panel 1: Primary Sensors (Ion Channels)",
    MECH_FIGDIR / "mechanosensing_panel1_primary_sensors.pdf",
    figsize=(30, 20)
)

# Panel 2: Surface Receptors
print("\nPanel 2: Surface Receptors")
plot_mechanosensing_panel(
    mech_dotplot_data,
    PANEL_2_GROUPS,
    "Mechanosensing Panel 2: Surface Receptors (Integrins & Shear Sensors)",
    MECH_FIGDIR / "mechanosensing_panel2_surface_receptors.pdf",
    figsize=(30, 20)
)

# Panel 3: Cellular Transduction
print("\nPanel 3: Cellular Mechanotransduction")
plot_mechanosensing_panel(
    mech_dotplot_data,
    PANEL_3_GROUPS,
    "Mechanosensing Panel 3: Cellular Transduction (FA, AJ, Membrane)",
    MECH_FIGDIR / "mechanosensing_panel3_cellular_transduction.pdf",
    figsize=(30, 20)
)

# Panel 4: Nuclear & ECM
print("\nPanel 4: Nuclear Mechanotransduction & ECM")
plot_mechanosensing_panel(
    mech_dotplot_data,
    PANEL_4_GROUPS,
    "Mechanosensing Panel 4: Nuclear & ECM (LINC, YAP/TAZ, ECM Ligands)",
    MECH_FIGDIR / "mechanosensing_panel4_nuclear_ECM.pdf",
    figsize=(30, 20)
)

print("\n" + "="*80)
print("MECHANOSENSING DOTPLOT GENERATION COMPLETE")
print("="*80)
print(f"\nOutput directory: {MECH_FIGDIR}")
print(f"Files generated:")
for f in sorted(MECH_FIGDIR.glob("*.pdf")):
    print(f"  - {f.name}")


[STEP] Generating mechanosensing dotplots (4 panels)...

Panel 1: Primary Mechanosensitive Sensors
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/mechanosensing_panel1_primary_sensors.pdf

Panel 2: Surface Receptors
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/mechanosensing_panel2_surface_receptors.pdf

Panel 3: Cellular Mechanotransduction
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/mechanosensing_panel3_cellular_transduction.pdf

Panel 4: Nuclear Mechanotransduction & ECM
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/mechanosensing_panel4_nuclear_ECM.pdf

MECHANOSENSING DOTPLOT GENERATION COMPLETE

Output directory: /home/h2048/data/py/1128/bbknn_annotation_analysis

In [ ]:
# ========== OPTIMIZED: Larger Dots for Better Visibility ==========

def plot_mechanosensing_panel_LARGE(data, groups, title, filename, figsize=(30, 20)):
    """
    Generate mechanosensing dotplot with LARGE, visible dots.
    
    Optimizations:
    - Much larger dot sizes (3x-10x scaling)
    - Increased font sizes
    - Better spacing
    - Optimized for large figure output
    """
    import matplotlib.patches as mpatches
    from matplotlib.colors import LinearSegmentedColormap
    
    # Filter data for this panel
    panel_data = data[data['group'].isin(groups)].copy()
    panel_data['group'] = pd.Categorical(
        panel_data['group'],
        categories=groups,
        ordered=True
    )
    
    if len(panel_data) == 0:
        print(f"  ⚠️  No data for panel: {title}")
        return None
    
    # Custom colormap (blue to red)
    colors_list = [
        "#1E466E", "#376795", "#528FAD", "#72BCD5", "#AADCE0",
        "#FFE6B7", "#FFD06F", "#F7AA58", "#EF8A47", "#E76254"
    ]
    cmap = LinearSegmentedColormap.from_list("custom_cmap", colors_list, N=256)
    
    # Create figure with subplots
    n_groups = len(groups)
    fig, axes = plt.subplots(1, n_groups, figsize=figsize, 
                            gridspec_kw={'wspace': 0.1})
    
    if n_groups == 1:
        axes = [axes]
    
    for idx, (ax, group) in enumerate(zip(axes, groups)):
        group_data = panel_data[panel_data['group'] == group]
        
        if len(group_data) == 0:
            ax.set_visible(False)
            continue
        
        # Pivot for plotting
        expr_pivot = group_data.pivot(index='cell_type', columns='gene', values='mean_expression')
        pct_pivot = group_data.pivot(index='cell_type', columns='gene', values='pct_expressed')
        
        # Get unique genes in original order
        genes_in_group = [g for g in present_mech_genes if g in expr_pivot.columns]
        cell_types = expr_pivot.index.tolist()
        
        # ========== CRITICAL: MUCH LARGER DOT SIZES ==========
        # Calculate adaptive base size based on figure dimensions
        n_genes = len(genes_in_group)
        n_celltypes = len(cell_types)
        
        # Adaptive sizing: smaller grids get bigger dots
        base_size_factor = min(3000 / (n_genes * n_celltypes), 50)
        
        # Plot dots with LARGE sizes
        for i, ct in enumerate(cell_types):
            for j, gene in enumerate(genes_in_group):
                if pd.notna(expr_pivot.loc[ct, gene]):
                    # Size scaled by percentage (0-100%) + large base factor
                    pct = pct_pivot.loc[ct, gene]

                    # 1) auto-fix scale: if pct is fraction (0-1), convert to 0-100
                    if pct <= 1.0:
                        pct = pct * 100.0

                    # 2) normalize within each gene to expand contrast even when absolute pct is tiny
                    gene_max = np.nanmax(pct_pivot[gene].values)
                    if gene_max <= 1.0:   # also handle fraction in the whole column
                        gene_max = gene_max * 100.0

                    pct_norm = 0.0 if gene_max <= 0 else pct / gene_max   # 0..1

                    # 3) nonlinear scaling: GAMMA < 1 boosts low pct differences
                    MIN_SIZE = 30
                    MAX_SIZE = 1000
                    GAMMA = 0.35
                    size = MIN_SIZE + (pct_norm ** GAMMA) * (MAX_SIZE - MIN_SIZE)


                    color = expr_pivot.loc[ct, gene]
                    
                    ax.scatter(j, i, s=size, c=[color], cmap=cmap,
                             vmin=-2, vmax=2, 
                             edgecolors='black', linewidths=1.5,  # Thicker borders
                             alpha=0.95)
        
        # Formatting
        ax.set_xlim(-0.5, len(genes_in_group) - 0.5)
        ax.set_ylim(-0.5, len(cell_types) - 0.5)
        ax.set_xticks(range(len(genes_in_group)))
        ax.set_xticklabels(genes_in_group, rotation=45, ha='right', 
                          fontsize=16, fontweight='normal')  # Larger font
        
        if idx == 0:
            ax.set_yticks(range(len(cell_types)))
            ax.set_yticklabels(cell_types, fontsize=16)  # Larger font
        else:
            ax.set_yticks([])
        
        ax.set_title(group, fontsize=18, fontweight='bold', pad=15)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.grid(False)
    
    # Overall title
    fig.suptitle(title, fontsize=22, fontweight='bold', y=0.98)
    
    # ========== LARGER COLORBAR ==========
    sm_expr = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=-2, vmax=2))
    sm_expr.set_array([])
    cbar_ax_expr = fig.add_axes([0.92, 0.55, 0.015, 0.3])
    cbar_expr = fig.colorbar(sm_expr, cax=cbar_ax_expr)
    cbar_expr.set_label('Scaled Expression', rotation=270, labelpad=25, fontsize=16)
    cbar_expr.ax.tick_params(labelsize=14)
    
    # ========== LARGER SIZE LEGEND ==========
    # Use actual representative sizes
    legend_sizes = [25, 50, 75, 100]
    size_legend_elements = []
    for pct in legend_sizes:
        # Match the actual size calculation
        actual_size = MIN_SIZE + (pct / 100.0) ** GAMMA * (MAX_SIZE - MIN_SIZE)
        # Convert to legend radius (sqrt scaling for area)
        radius = np.sqrt(actual_size) / 80  # Adjusted scaling for legend
        size_legend_elements.append(
            mpatches.Circle((0, 0), radius=radius, 
                          facecolor='gray', edgecolor='black', 
                          linewidth=1.5, label=f'{pct}%')
        )
    
    legend = fig.legend(handles=size_legend_elements, 
                       loc='center right',
                       bbox_to_anchor=(0.99, 0.25), 
                       title='% Expressed',
                       frameon=False, 
                       fontsize=14,
                       title_fontsize=16)
    plt.setp(legend.get_title(), fontweight='bold')
    
    # Save with high DPI
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  ✓ Saved: {filename}")
    plt.close()
    
    return fig


# ========== RE-GENERATE WITH LARGER DOTS ==========
print("\n[STEP] Re-generating mechanosensing dotplots with LARGER dots...\n")

# Panel 1: Primary Sensors
print("Panel 1: Primary Mechanosensitive Sensors")
plot_mechanosensing_panel_LARGE(
    mech_dotplot_data,
    PANEL_1_GROUPS,
    "Mechanosensing Panel 1: Primary Sensors (Ion Channels)",
    MECH_FIGDIR / "mechanosensing_panel1_primary_sensors_LARGE.pdf",
    figsize=(32, 40)
)

# Panel 2: Surface Receptors
print("\nPanel 2: Surface Receptors")
plot_mechanosensing_panel_LARGE(
    mech_dotplot_data,
    PANEL_2_GROUPS,
    "Mechanosensing Panel 2: Surface Receptors (Integrins & Shear Sensors)",
    MECH_FIGDIR / "mechanosensing_panel2_surface_receptors_LARGE.pdf",
    figsize=(36, 40)
)

# Panel 3: Cellular Transduction
print("\nPanel 3: Cellular Mechanotransduction")
plot_mechanosensing_panel_LARGE(
    mech_dotplot_data,
    PANEL_3_GROUPS,
    "Mechanosensing Panel 3: Cellular Transduction (FA, AJ, Membrane)",
    MECH_FIGDIR / "mechanosensing_panel3_cellular_transduction_LARGE.pdf",
    figsize=(40, 40)
)

# Panel 4: Nuclear & ECM
print("\nPanel 4: Nuclear Mechanotransduction & ECM")
plot_mechanosensing_panel_LARGE(
    mech_dotplot_data,
    PANEL_4_GROUPS,
    "Mechanosensing Panel 4: Nuclear & ECM (LINC, YAP/TAZ, ECM Ligands)",
    MECH_FIGDIR / "mechanosensing_panel4_nuclear_ECM_LARGE.pdf",
    figsize=(34, 40)
)

print("\n" + "="*80)
print("LARGE DOT MECHANOSENSING DOTPLOTS COMPLETE")
print("="*80)
print(f"\nOutput directory: {MECH_FIGDIR}")
print(f"New files generated (LARGE version):")
for f in sorted(MECH_FIGDIR.glob("*_LARGE.pdf")):
    print(f"  - {f.name}")


[STEP] Re-generating mechanosensing dotplots with LARGER dots...

Panel 1: Primary Mechanosensitive Sensors
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/mechanosensing_panel1_primary_sensors_LARGE.pdf

Panel 2: Surface Receptors
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/mechanosensing_panel2_surface_receptors_LARGE.pdf

Panel 3: Cellular Mechanotransduction
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/mechanosensing_panel3_cellular_transduction_LARGE.pdf

Panel 4: Nuclear Mechanotransduction & ECM
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/mechanosensing_panel4_nuclear_ECM_LARGE.pdf

LARGE DOT MECHANOSENSING DOTPLOTS COMPLETE

Output directory: /home/h2048/data/

In [49]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import issparse

# ---------- auto detect tissue key ----------
CANDIDATES = ["tissue", "tissue_type", "Tissue", "anatomical_region", "region", "site"]
TISSUE_KEY = next((k for k in CANDIDATES if k in main.obs.columns), None)
if TISSUE_KEY is None:
    raise KeyError(f"Cannot find tissue key in main.obs. Tried: {CANDIDATES}")
if not pd.api.types.is_categorical_dtype(main.obs[TISSUE_KEY]):
    main.obs[TISSUE_KEY] = main.obs[TISSUE_KEY].astype("category")

print("Using TISSUE_KEY =", TISSUE_KEY)

# ---------- make sure panel group lists exist ----------
# 你已有 PANEL_1_GROUPS... 就跳过；否则用 PANEL_1..4 或 MECH_PRIORITY 自动兜底
if "PANEL_1_GROUPS" not in globals():
    if "PANEL_1" in globals():
        PANEL_1_GROUPS, PANEL_2_GROUPS, PANEL_3_GROUPS, PANEL_4_GROUPS = PANEL_1, PANEL_2, PANEL_3, PANEL_4
    else:
        # 最保底：按 MECH_PRIORITY 切 4 份
        def _chunk(groups, n=4):
            groups = list(groups)
            cuts = np.linspace(0, len(groups), n+1).round().astype(int)
            return [groups[cuts[i]:cuts[i+1]] for i in range(n)]
        PANEL_1_GROUPS, PANEL_2_GROUPS, PANEL_3_GROUPS, PANEL_4_GROUPS = _chunk(MECH_PRIORITY, 4)

# ---------- build mech_dotplot_data by tissue ----------
def build_mech_dotplot_data_by_tissue(
    adata,
    celltype_key,
    tissue_key,
    genes,
    mech_gene_sets,
    use_raw=True,
    zscore_within_tissue=True,
    min_cells_per_tissue=50
):
    rows = []
    use_raw = (use_raw and (adata.raw is not None))
    gene_universe = adata.raw.var_names if use_raw else adata.var_names
    genes = [g for g in genes if g in gene_universe]
    if len(genes) == 0:
        raise ValueError("No genes present in gene universe (raw.var_names or var_names).")

    # ensure categorical
    if not pd.api.types.is_categorical_dtype(adata.obs[celltype_key]):
        adata.obs[celltype_key] = adata.obs[celltype_key].astype("category")
    if not pd.api.types.is_categorical_dtype(adata.obs[tissue_key]):
        adata.obs[tissue_key] = adata.obs[tissue_key].astype("category")

    tissues = adata.obs[tissue_key].cat.categories.tolist()
    for tis in tissues:
        mask_t = (adata.obs[tissue_key] == tis).values
        if mask_t.sum() < min_cells_per_tissue:
            continue

        ad_t = adata[mask_t]
        cts = ad_t.obs[celltype_key].cat.categories.tolist()

        for ct in cts:
            mask_ct = (ad_t.obs[celltype_key] == ct).values
            if mask_ct.sum() == 0:
                continue
            ad_tc = ad_t[mask_ct]

            X = (ad_tc.raw[:, genes].X if use_raw else ad_tc[:, genes].X)

            if issparse(X):
                mean_expr = np.asarray(X.mean(axis=0)).ravel()
                pct_expr = np.asarray((X > 0).mean(axis=0)).ravel() * 100.0
            else:
                X = np.asarray(X)
                mean_expr = X.mean(axis=0).ravel()
                pct_expr = (X > 0).mean(axis=0).ravel() * 100.0

            for g, m, p in zip(genes, mean_expr, pct_expr):
                # 找到 gene 属于哪个 group（按 mech_gene_sets 第一个命中）
                g_group = None
                for grp, glist in mech_gene_sets.items():
                    if g in glist:
                        g_group = grp
                        break
                if g_group is None:
                    g_group = "UNASSIGNED"

                rows.append({
                    "tissue": tis,
                    "cell_type": ct,
                    "gene": g,
                    "group": g_group,
                    "mean_expression_raw": float(m),
                    "pct_expressed": float(p),
                })

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No dotplot rows generated (check min_cells_per_tissue / keys / genes).")

    # z-score（让颜色落在 -2~2 这种范围更舒服）
    if zscore_within_tissue:
        df["mean_expression"] = df.groupby(["tissue", "gene"])["mean_expression_raw"].transform(
            lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-10)
        )
    else:
        df["mean_expression"] = df.groupby(["gene"])["mean_expression_raw"].transform(
            lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-10)
        )

    return df

# 生成
mech_dotplot_data_by_tissue = build_mech_dotplot_data_by_tissue(
    main,
    celltype_key=MAIN_L3_KEY,
    tissue_key=TISSUE_KEY,
    genes=present_mech_genes,      # 你已有的 present_mech_genes
    mech_gene_sets=MECH_GENE_SETS,
    use_raw=True,
    zscore_within_tissue=True,     # 想跨 tissue 可比就改 False
    min_cells_per_tissue=50
)

print(mech_dotplot_data_by_tissue.shape)


Using TISSUE_KEY = tissue


(58888, 7)


In [50]:
import re

MECH_FIGDIR_BY_TISSUE = Path(MECH_FIGDIR) / "by_tissue"
MECH_FIGDIR_BY_TISSUE.mkdir(parents=True, exist_ok=True)

tissues = mech_dotplot_data_by_tissue["tissue"].unique().tolist()
print("Tissues to plot:", tissues)

for tis in tissues:
    df_t = mech_dotplot_data_by_tissue[mech_dotplot_data_by_tissue["tissue"] == tis].copy()

    tis_safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(tis))
    outdir = MECH_FIGDIR_BY_TISSUE / tis_safe
    outdir.mkdir(parents=True, exist_ok=True)

    print("\n====", tis, "====")

    plot_mechanosensing_panel_LARGE(
        df_t, PANEL_1_GROUPS,
        f"{tis} | Panel 1: Primary Sensors",
        outdir / f"{tis_safe}_panel1_PRIMARY_LARGE.pdf",
        figsize=(32, 22)
    )
    plot_mechanosensing_panel_LARGE(
        df_t, PANEL_2_GROUPS,
        f"{tis} | Panel 2: Surface Receptors",
        outdir / f"{tis_safe}_panel2_SURFACE_LARGE.pdf",
        figsize=(36, 22)
    )
    plot_mechanosensing_panel_LARGE(
        df_t, PANEL_3_GROUPS,
        f"{tis} | Panel 3: Cellular Transduction",
        outdir / f"{tis_safe}_panel3_CELLULAR_LARGE.pdf",
        figsize=(40, 22)
    )
    plot_mechanosensing_panel_LARGE(
        df_t, PANEL_4_GROUPS,
        f"{tis} | Panel 4: Nuclear & ECM",
        outdir / f"{tis_safe}_panel4_NUCLEAR_ECM_LARGE.pdf",
        figsize=(34, 22)
    )

print("\nDONE. Output:", MECH_FIGDIR_BY_TISSUE)


Tissues to plot: ['lung parenchyma', 'nose', 'respiratory airway', 'sinus']

==== lung parenchyma ====
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/by_tissue/lung_parenchyma/lung_parenchyma_panel1_PRIMARY_LARGE.pdf
  ✓ Saved: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/by_tissue/lung_parenchyma/lung_parenchyma_panel2_SURFACE_LARGE.pdf


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# mech_dotplot_data_by_tissue: 需要至少包含列
# ['tissue','cell_type','gene','group','mean_expression_raw','pct_expressed']
df = mech_dotplot_data_by_tissue.copy()

# ---- 1) 固定 4 个 tissue（如果不止 4 个，默认取细胞数最多的 4 个；你也可以手动指定）----
if "TISSUES_4" not in globals():
    top4 = (main.obs[TISSUE_KEY].value_counts().head(4).index.tolist()
            if "TISSUE_KEY" in globals() else
            df["tissue"].value_counts().head(4).index.tolist())
    TISSUES_4 = top4

print("TISSUES_4 =", TISSUES_4)

df = df[df["tissue"].isin(TISSUES_4)].copy()
df["tissue"] = pd.Categorical(df["tissue"], categories=TISSUES_4, ordered=True)

# ---- 2) 颜色值：建议 gene-wise 全局 zscore（跨 tissue 可比）----
if "mean_expression" not in df.columns or df["mean_expression"].isna().all():
    df["mean_expression"] = df.groupby("gene")["mean_expression_raw"].transform(
        lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-10)
    )

# ---- 3) 输出目录 ----
OUTDIR = Path(MECH_FIGDIR) / "by_group_4tissue_gene_blocks"
OUTDIR.mkdir(parents=True, exist_ok=True)
print("OUTDIR =", OUTDIR)

# ---- 4) cell_type 顺序（用主对象类别顺序；否则按字母）----
if "MAIN_L3_KEY" in globals() and pd.api.types.is_categorical_dtype(main.obs[MAIN_L3_KEY]):
    CELLTYPE_ORDER = main.obs[MAIN_L3_KEY].cat.categories.tolist()
else:
    CELLTYPE_ORDER = sorted(df["cell_type"].unique().tolist())


TISSUES_4 = ['nose', 'lung parenchyma', 'sinus', 'respiratory airway']
OUTDIR = /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/by_group_4tissue_gene_blocks


In [54]:
# ================================================================================
# CELL 2/2 — DOTPLOT: each group one (or multiple parts) figure
# Layout: gene blocks × 4 tissues side-by-side, y = cell_type
# FIXED: prevent "Image size too large" by capping figsize + auto DPI downscale
# ================================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import re

# -----------------------------
# 0) Output dir
# -----------------------------
OUTDIR_DOT = Path(OUTDIR) / "dotplots_gene_blocks"
OUTDIR_DOT.mkdir(parents=True, exist_ok=True)
print("OUTDIR_DOT =", OUTDIR_DOT)

# -----------------------------
# 1) Colormap (blue->red)
# -----------------------------
colors_list = [
    "#1E466E", "#376795", "#528FAD", "#72BCD5", "#AADCE0",
    "#FFE6B7", "#FFD06F", "#F7AA58", "#EF8A47", "#E76254"
]
cmap = LinearSegmentedColormap.from_list("mech_cmap", colors_list, N=256)

# -----------------------------
# 2) Safe savefig (auto downscale DPI to avoid >65536 px)
# -----------------------------
def _safe_savefig(fig, path, dpi=300, max_px=65000, **kwargs):
    w_in, h_in = fig.get_size_inches()
    # dpi must satisfy both width and height constraints
    dpi_w = int((max_px - 1) / max(w_in, 1e-9))
    dpi_h = int((max_px - 1) / max(h_in, 1e-9))
    dpi_safe = min(dpi, dpi_w, dpi_h)
    dpi_safe = max(72, dpi_safe)  # don't go too low
    fig.savefig(path, dpi=dpi_safe, **kwargs)
    return dpi_safe

# -----------------------------
# 3) Main plotting function
# -----------------------------
def plot_group_gene_blocks_4tissue(
    df_group,
    group_name,
    out_pdf,
    gene_order=None,
    celltype_order=None,
    tissues=None,
    gap=0.8,                   # gene-block gap in x units
    # auto figsize control
    max_fig_w=120,             # ✅ hard cap width (inches)
    min_fig_w=22,
    base_fig_h=18,
    # color scale
    vmin=-2, vmax=2,
    # dot size mapping
    min_size=40,
    max_size=3200,
    gamma=0.45,                # <1 boosts low pct differences
):
    if tissues is None:
        tissues = df_group["tissue"].cat.categories.tolist() if pd.api.types.is_categorical_dtype(df_group["tissue"]) else sorted(df_group["tissue"].unique())
    n_t = len(tissues)
    if n_t == 0:
        print(f"  ⚠️ skip {group_name}: no tissues")
        return

    # orders
    if gene_order is None:
        gene_order = [g for g in present_mech_genes if g in df_group["gene"].unique()]
        if len(gene_order) == 0:
            gene_order = sorted(df_group["gene"].unique().tolist())
    if celltype_order is None:
        celltype_order = CELLTYPE_ORDER

    df_group = df_group.copy()
    df_group = df_group[df_group["gene"].isin(gene_order)]
    df_group = df_group[df_group["cell_type"].isin(celltype_order)]
    df_group = df_group[df_group["tissue"].isin(tissues)]
    if df_group.empty:
        print(f"  ⚠️ skip {group_name}: empty after filtering")
        return

    # coordinate maps
    gene_to_i = {g:i for i,g in enumerate(gene_order)}
    tissue_to_j = {t:j for j,t in enumerate(tissues)}
    cell_to_y = {ct:i for i,ct in enumerate(celltype_order)}

    # prepare scatter arrays
    xs, ys, cs, ss = [], [], [], []

    # ✅ per-gene normalization of pct to avoid "all same size" when pct range is tiny
    for g in gene_order:
        subg = df_group[df_group["gene"] == g]
        if subg.empty:
            continue

        pct_col = subg["pct_expressed"].astype(float).values
        # auto-fix fraction (0-1) -> percent (0-100)
        if np.nanmax(pct_col) <= 1.0:
            pct_col = pct_col * 100.0
        gmax = np.nanmax(pct_col)
        if not np.isfinite(gmax) or gmax <= 0:
            gmax = 100.0

        for _, r in subg.iterrows():
            ct = r["cell_type"]
            t = r["tissue"]
            if (ct not in cell_to_y) or (t not in tissue_to_j):
                continue

            pct = float(r["pct_expressed"])
            if pct <= 1.0:  # fraction
                pct *= 100.0
            pct_norm = np.clip(pct / gmax, 0, 1)

            x = gene_to_i[g] * (n_t + gap) + tissue_to_j[t]
            y = cell_to_y[ct]

            xs.append(x)
            ys.append(y)
            cs.append(float(r["mean_expression"]))
            ss.append(min_size + (pct_norm ** gamma) * (max_size - min_size))

    if len(xs) == 0:
        print(f"  ⚠️ skip {group_name}: no points")
        return

    # -----------------------------
    # auto figsize (bounded)
    # -----------------------------
    n_genes = len(gene_order)
    x_span = n_genes * (n_t + gap)  # approximate total x span
    fig_w = max(min_fig_w, 0.65 * x_span + 8)   # heuristic
    fig_w = min(max_fig_w, fig_w)               # ✅ cap width hard
    fig_h = base_fig_h

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sca = ax.scatter(
        xs, ys,
        s=ss,
        c=cs,
        cmap=cmap,
        vmin=vmin, vmax=vmax,
        edgecolors="black",
        linewidths=0.8,
        alpha=0.95
    )

    # y axis
    ax.set_yticks(range(len(celltype_order)))
    ax.set_yticklabels(celltype_order, fontsize=11)
    ax.invert_yaxis()

    # bottom x: tissue labels repeated per gene block
    xticks, xticklabels = [], []
    for gi, g in enumerate(gene_order):
        base = gi * (n_t + gap)
        for t in tissues:
            xticks.append(base + tissue_to_j[t])
            xticklabels.append(str(t))
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=90, fontsize=9)

    # top axis: gene labels centered per block
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    gene_centers = [gi * (n_t + gap) + (n_t - 1) / 2 for gi in range(len(gene_order))]
    ax2.set_xticks(gene_centers)
    ax2.set_xticklabels(gene_order, fontsize=11, fontweight="bold")
    ax2.tick_params(axis="x", length=0)

    # separators between genes + light alternating background blocks
    for gi in range(len(gene_order)):
        start = gi * (n_t + gap) - 0.5
        end = gi * (n_t + gap) + (n_t - 1) + 0.5
        if gi % 2 == 0:
            ax.axvspan(start, end, color="black", alpha=0.03, zorder=0)
        if gi > 0:
            ax.axvline(start + 0.5, color="black", linewidth=1.0, alpha=0.6)

    ax.set_title(f"{group_name} — gene blocks × {n_t} tissues", fontsize=14, fontweight="bold", pad=18)
    ax.set_xlabel("")
    ax.set_ylabel("Cell Type", fontsize=12)

    # colorbar
    cbar = fig.colorbar(sca, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label("Scaled expression (gene-wise z)", rotation=270, labelpad=14, fontsize=10)

    plt.tight_layout()

    # ✅ safe save (auto DPI downscale)
    dpi_used = _safe_savefig(fig, out_pdf, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✓ Saved: {Path(out_pdf).name} (figsize={fig_w:.1f}x{fig_h:.1f} in, dpi={dpi_used})")


# -----------------------------
# 4) Batch export: each group (auto chunk if too many genes)
# -----------------------------
MAX_GENES_PER_FIG = 14  # ✅ 推荐 10-15；genes多时自动拆part，避免单张过宽/不可读

groups_to_plot = [g for g in MECH_PRIORITY if g in df["group"].unique()]
print("Groups to plot:", len(groups_to_plot))
print("Tissues:", TISSUES_4)

for gname in groups_to_plot:
    sub = df[df["group"] == gname].copy()
    if sub.empty:
        continue

    gene_order = [x for x in MECH_GENE_SETS.get(gname, []) if x in sub["gene"].unique()]
    if len(gene_order) == 0:
        continue

    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(gname))

    # split into parts if too many genes
    if len(gene_order) > MAX_GENES_PER_FIG:
        for part_i, start in enumerate(range(0, len(gene_order), MAX_GENES_PER_FIG), 1):
            genes_part = gene_order[start:start + MAX_GENES_PER_FIG]
            out_pdf = OUTDIR_DOT / f"group_{safe}_part{part_i:02d}_DOT_4tissue_gene_blocks.pdf"
            plot_group_gene_blocks_4tissue(
                sub,
                group_name=f"{gname} (part {part_i:02d})",
                out_pdf=out_pdf,
                gene_order=genes_part,
                celltype_order=CELLTYPE_ORDER,
                tissues=TISSUES_4,
                min_size=40,
                max_size=3200,
                gamma=0.45,
                max_fig_w=120,   # ✅ hard cap
                base_fig_h=18
            )
    else:
        out_pdf = OUTDIR_DOT / f"group_{safe}_DOT_4tissue_gene_blocks.pdf"
        plot_group_gene_blocks_4tissue(
            sub,
            group_name=gname,
            out_pdf=out_pdf,
            gene_order=gene_order,
            celltype_order=CELLTYPE_ORDER,
            tissues=TISSUES_4,
            min_size=40,
            max_size=3200,
            gamma=0.45,
            max_fig_w=120,     # ✅ hard cap
            base_fig_h=18
        )

print("\nDONE:", OUTDIR_DOT)


OUTDIR_DOT = /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/by_group_4tissue_gene_blocks/dotplots_gene_blocks
Groups to plot: 15
Tissues: ['nose', 'lung parenchyma', 'sinus', 'respiratory airway']
  ✓ Saved: group_Classic_mechanosensitive_channels_DOT_4tissue_gene_blocks.pdf (figsize=22.0x18.0 in, dpi=300)
  ✓ Saved: group_TMC_MET_complex_DOT_4tissue_gene_blocks.pdf (figsize=48.6x18.0 in, dpi=300)
  ✓ Saved: group_TRP-related_mechanosensors_DOT_4tissue_gene_blocks.pdf (figsize=23.6x18.0 in, dpi=300)
  ✓ Saved: group_K2P_stretch-sensitive_K_channels_DOT_4tissue_gene_blocks.pdf (figsize=22.0x18.0 in, dpi=300)
  ✓ Saved: group_ENaC_ASIC_family_DOT_4tissue_gene_blocks.pdf (figsize=26.7x18.0 in, dpi=300)
  ✓ Saved: group_Integrins_ECM_force_receptors__part01_DOT_4tissue_gene_blocks.pdf (figsize=51.7x18.0 in, dpi=300)
  ✓ Saved: group_Integrins_ECM_force_receptors__part02_DOT_4tissue_gene_blocks.pdf (figsize=42.3x18.0 in, dp

In [55]:
# ================================================================================
# CELL 2/2 — DOTPLOT (DEFAULT dot size + TALLER figure)
# Layout: gene blocks × 4 tissues side-by-side, y = cell_type
# FIXED: auto taller height, safe savefig, auto chunk genes
# ================================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import re

# -----------------------------
# 0) Output dir
# -----------------------------
OUTDIR_DOT = Path(OUTDIR) / "dotplots_gene_blocks"
OUTDIR_DOT.mkdir(parents=True, exist_ok=True)
print("OUTDIR_DOT =", OUTDIR_DOT)

# -----------------------------
# 1) Colormap (blue->red)
# -----------------------------
colors_list = [
    "#1E466E", "#376795", "#528FAD", "#72BCD5", "#AADCE0",
    "#FFE6B7", "#FFD06F", "#F7AA58", "#EF8A47", "#E76254"
]
cmap = LinearSegmentedColormap.from_list("mech_cmap", colors_list, N=256)

# -----------------------------
# 2) Safe savefig (auto downscale DPI to avoid >65536 px)
# -----------------------------
def _safe_savefig(fig, path, dpi=300, max_px=65000, **kwargs):
    w_in, h_in = fig.get_size_inches()
    dpi_w = int((max_px - 1) / max(w_in, 1e-9))
    dpi_h = int((max_px - 1) / max(h_in, 1e-9))
    dpi_safe = min(dpi, dpi_w, dpi_h)
    dpi_safe = max(72, dpi_safe)
    fig.savefig(path, dpi=dpi_safe, **kwargs)
    return dpi_safe

# -----------------------------
# 3) Main plotting function (DEFAULT dot size + auto taller height)
# -----------------------------
def plot_group_gene_blocks_4tissue(
    df_group,
    group_name,
    out_pdf,
    gene_order=None,
    celltype_order=None,
    tissues=None,
    gap=0.8,                      # gene-block gap in x units
    # figsize control
    max_fig_w=120, min_fig_w=22,
    min_fig_h=18, max_fig_h=42,   # ✅ allow taller
    row_inch=0.28,                # ✅ height per celltype (tune: 0.22~0.35)
    # color scale
    vmin=-2, vmax=2,
    # dot size mapping (DEFAULT-ish)
    min_size=12,                  # ✅ normal dot baseline
    max_size=520,                 # ✅ normal max area (was 3200 in LARGE mode)
    gamma=0.85,                   # ✅ closer to linear; lower=more exaggeration
):
    if tissues is None:
        tissues = (df_group["tissue"].cat.categories.tolist()
                   if pd.api.types.is_categorical_dtype(df_group["tissue"])
                   else sorted(df_group["tissue"].unique()))
    n_t = len(tissues)
    if n_t == 0:
        print(f"  ⚠️ skip {group_name}: no tissues")
        return

    if gene_order is None:
        gene_order = [g for g in present_mech_genes if g in df_group["gene"].unique()]
        if len(gene_order) == 0:
            gene_order = sorted(df_group["gene"].unique().tolist())
    if celltype_order is None:
        celltype_order = CELLTYPE_ORDER

    df_group = df_group.copy()
    df_group = df_group[df_group["gene"].isin(gene_order)]
    df_group = df_group[df_group["cell_type"].isin(celltype_order)]
    df_group = df_group[df_group["tissue"].isin(tissues)]
    if df_group.empty:
        print(f"  ⚠️ skip {group_name}: empty after filtering")
        return

    gene_to_i = {g:i for i,g in enumerate(gene_order)}
    tissue_to_j = {t:j for j,t in enumerate(tissues)}
    cell_to_y = {ct:i for i,ct in enumerate(celltype_order)}

    xs, ys, cs, ss = [], [], [], []

    # per-gene normalization of pct (keeps gradient even if absolute pct small)
    for g in gene_order:
        subg = df_group[df_group["gene"] == g]
        if subg.empty:
            continue

        pct_vec = subg["pct_expressed"].astype(float).values
        if np.nanmax(pct_vec) <= 1.0:  # fraction -> %
            pct_vec = pct_vec * 100.0
        gmax = np.nanmax(pct_vec)
        if not np.isfinite(gmax) or gmax <= 0:
            gmax = 100.0

        for _, r in subg.iterrows():
            ct = r["cell_type"]
            t = r["tissue"]
            if (ct not in cell_to_y) or (t not in tissue_to_j):
                continue

            pct = float(r["pct_expressed"])
            if pct <= 1.0:
                pct *= 100.0
            pct_norm = np.clip(pct / gmax, 0, 1)

            x = gene_to_i[g] * (n_t + gap) + tissue_to_j[t]
            y = cell_to_y[ct]

            xs.append(x)
            ys.append(y)
            cs.append(float(r["mean_expression"]))
            ss.append(min_size + (pct_norm ** gamma) * (max_size - min_size))

    if len(xs) == 0:
        print(f"  ⚠️ skip {group_name}: no points")
        return

    # -----------------------------
    # auto figsize: bounded width + ✅ taller height based on #celltypes
    # -----------------------------
    n_genes = len(gene_order)
    x_span = n_genes * (n_t + gap)
    fig_w = max(min_fig_w, 0.65 * x_span + 8)
    fig_w = min(max_fig_w, fig_w)

    n_ct = len(celltype_order)
    fig_h = max(min_fig_h, 6 + row_inch * n_ct)   # ✅ taller as ct increases
    fig_h = min(max_fig_h, fig_h)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sca = ax.scatter(
        xs, ys,
        s=ss,
        c=cs,
        cmap=cmap,
        vmin=vmin, vmax=vmax,
        edgecolors="black",
        linewidths=0.6,
        alpha=0.95
    )

    # y axis
    ax.set_yticks(range(n_ct))
    ax.set_yticklabels(celltype_order, fontsize=11)
    ax.invert_yaxis()

    # bottom x: tissue labels repeated per gene block
    xticks, xticklabels = [], []
    for gi, g in enumerate(gene_order):
        base = gi * (n_t + gap)
        for t in tissues:
            xticks.append(base + tissue_to_j[t])
            xticklabels.append(str(t))
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=90, fontsize=9)

    # top axis: gene labels centered per block
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    gene_centers = [gi * (n_t + gap) + (n_t - 1) / 2 for gi in range(n_genes)]
    ax2.set_xticks(gene_centers)
    ax2.set_xticklabels(gene_order, fontsize=11, fontweight="bold")
    ax2.tick_params(axis="x", length=0)

    # separators between genes + light background blocks
    for gi in range(n_genes):
        start = gi * (n_t + gap) - 0.5
        end = gi * (n_t + gap) + (n_t - 1) + 0.5
        if gi % 2 == 0:
            ax.axvspan(start, end, color="black", alpha=0.03, zorder=0)
        if gi > 0:
            ax.axvline(start + 0.5, color="black", linewidth=1.0, alpha=0.6)

    ax.set_title(f"{group_name} — gene blocks × {n_t} tissues", fontsize=14, fontweight="bold", pad=18)
    ax.set_xlabel("")
    ax.set_ylabel("Cell Type", fontsize=12)

    # colorbar
    cbar = fig.colorbar(sca, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label("Scaled expression (gene-wise z)", rotation=270, labelpad=14, fontsize=10)

    # ✅ 给 y 轴标签留更多空间，避免“放不下”
    plt.subplots_adjust(left=0.28, right=0.90, top=0.90, bottom=0.18)

    dpi_used = _safe_savefig(fig, out_pdf, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✓ Saved: {Path(out_pdf).name} (figsize={fig_w:.1f}x{fig_h:.1f} in, dpi={dpi_used})")


# -----------------------------
# 4) Batch export: each group (auto chunk if too many genes)
# -----------------------------
MAX_GENES_PER_FIG = 14  # genes多自动拆 part，避免过宽/不可读/爆像素

groups_to_plot = [g for g in MECH_PRIORITY if g in df["group"].unique()]
print("Groups to plot:", len(groups_to_plot))
print("Tissues:", TISSUES_4)

for gname in groups_to_plot:
    sub = df[df["group"] == gname].copy()
    if sub.empty:
        continue

    gene_order = [x for x in MECH_GENE_SETS.get(gname, []) if x in sub["gene"].unique()]
    if len(gene_order) == 0:
        continue

    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(gname))

    if len(gene_order) > MAX_GENES_PER_FIG:
        for part_i, start in enumerate(range(0, len(gene_order), MAX_GENES_PER_FIG), 1):
            genes_part = gene_order[start:start + MAX_GENES_PER_FIG]
            out_pdf = OUTDIR_DOT / f"group_{safe}_part{part_i:02d}_DOT_4tissue_gene_blocks.pdf"
            plot_group_gene_blocks_4tissue(
                sub,
                group_name=f"{gname} (part {part_i:02d})",
                out_pdf=out_pdf,
                gene_order=genes_part,
                celltype_order=CELLTYPE_ORDER,
                tissues=TISSUES_4,
            )
    else:
        out_pdf = OUTDIR_DOT / f"group_{safe}_DOT_4tissue_gene_blocks.pdf"
        plot_group_gene_blocks_4tissue(
            sub,
            group_name=gname,
            out_pdf=out_pdf,
            gene_order=gene_order,
            celltype_order=CELLTYPE_ORDER,
            tissues=TISSUES_4,
        )

print("\nDONE:", OUTDIR_DOT)


OUTDIR_DOT = /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/by_group_4tissue_gene_blocks/dotplots_gene_blocks
Groups to plot: 15
Tissues: ['nose', 'lung parenchyma', 'sinus', 'respiratory airway']
  ✓ Saved: group_Classic_mechanosensitive_channels_DOT_4tissue_gene_blocks.pdf (figsize=22.0x42.0 in, dpi=300)
  ✓ Saved: group_TMC_MET_complex_DOT_4tissue_gene_blocks.pdf (figsize=48.6x42.0 in, dpi=300)
  ✓ Saved: group_TRP-related_mechanosensors_DOT_4tissue_gene_blocks.pdf (figsize=23.6x42.0 in, dpi=300)
  ✓ Saved: group_K2P_stretch-sensitive_K_channels_DOT_4tissue_gene_blocks.pdf (figsize=22.0x42.0 in, dpi=300)
  ✓ Saved: group_ENaC_ASIC_family_DOT_4tissue_gene_blocks.pdf (figsize=26.7x42.0 in, dpi=300)
  ✓ Saved: group_Integrins_ECM_force_receptors__part01_DOT_4tissue_gene_blocks.pdf (figsize=51.7x42.0 in, dpi=300)
  ✓ Saved: group_Integrins_ECM_force_receptors__part02_DOT_4tissue_gene_blocks.pdf (figsize=42.3x42.0 in, dp

In [58]:
# ================================================================================
# CELL 2/2 — HEATMAP (DEFAULT size + TALLER figure) — FIX alpha NaN
# Layout: gene blocks × 4 tissues side-by-side, y = cell_type
# ================================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import re

OUTDIR_HM = Path(OUTDIR) / "heatmaps_gene_blocks"
OUTDIR_HM.mkdir(parents=True, exist_ok=True)
print("OUTDIR_HM =", OUTDIR_HM)

colors_list = [
    "#1E466E", "#376795", "#528FAD", "#72BCD5", "#AADCE0",
    "#FFE6B7", "#FFD06F", "#F7AA58", "#EF8A47", "#E76254"
]
cmap = LinearSegmentedColormap.from_list("mech_cmap", colors_list, N=256)
cmap2 = cmap.copy()
cmap2.set_bad(color="white")

def _safe_savefig(fig, path, dpi=300, max_px=65000, **kwargs):
    w_in, h_in = fig.get_size_inches()
    dpi_w = int((max_px - 1) / max(w_in, 1e-9))
    dpi_h = int((max_px - 1) / max(h_in, 1e-9))
    dpi_safe = min(dpi, dpi_w, dpi_h)
    dpi_safe = max(72, dpi_safe)
    fig.savefig(path, dpi=dpi_safe, **kwargs)
    return dpi_safe


def plot_group_heatmap_gene_blocks_4tissue(
    df_group,
    group_name,
    out_pdf,
    gene_order,
    celltype_order,
    tissues,
    gap_cols=1,
    max_fig_w=120, min_fig_w=22,
    min_fig_h=18, max_fig_h=46,
    row_inch=0.26,
    col_inch=0.32,
    vmin=-2, vmax=2,
    use_alpha_pct=True,
    alpha_gamma=0.55,
    alpha_min=0.18,
):
    df_group = df_group.copy()
    df_group = df_group[df_group["gene"].isin(gene_order)]
    df_group = df_group[df_group["cell_type"].isin(celltype_order)]
    df_group = df_group[df_group["tissue"].isin(tissues)]
    if df_group.empty:
        print(f"  ⚠️ skip {group_name}: empty after filtering")
        return

    # mean_expression matrix
    mat = df_group.pivot_table(
        index="cell_type",
        columns=["gene", "tissue"],
        values="mean_expression",
        aggfunc="mean"
    )
    full_cols = pd.MultiIndex.from_product([gene_order, tissues], names=["gene", "tissue"])
    mat = mat.reindex(index=celltype_order, columns=full_cols)

    n_ct = len(celltype_order)
    n_t = len(tissues)

    # alpha from pct_expressed (optional)
    alpha_data = None
    if use_alpha_pct:
        pct = df_group.pivot_table(
            index="cell_type",
            columns=["gene", "tissue"],
            values="pct_expressed",
            aggfunc="mean"
        ).reindex(index=celltype_order, columns=full_cols)

        pct_vals = pct.values.astype(float)
        if np.nanmax(pct_vals) <= 1.0:
            pct_vals = pct_vals * 100.0

        alpha_vals = np.full_like(pct_vals, np.nan, dtype=float)
        for gi, g in enumerate(gene_order):
            cols_g = slice(gi * n_t, (gi + 1) * n_t)
            block = pct_vals[:, cols_g]
            gmax = np.nanmax(block)
            if not np.isfinite(gmax) or gmax <= 0:
                gmax = 100.0
            block_norm = np.clip(block / gmax, 0, 1)
            alpha_vals[:, cols_g] = alpha_min + (block_norm ** alpha_gamma) * (1 - alpha_min)

        alpha_data = alpha_vals

    # expand with gap columns
    expanded_blocks = []
    expanded_alpha = [] if use_alpha_pct else None

    for gi, g in enumerate(gene_order):
        block = mat.loc[:, (g, tissues)].values  # (n_ct, n_t)
        expanded_blocks.append(block)
        if use_alpha_pct:
            expanded_alpha.append(alpha_data[:, gi * n_t:(gi + 1) * n_t])

        if gi < len(gene_order) - 1 and gap_cols > 0:
            expanded_blocks.append(np.full((n_ct, gap_cols), np.nan))
            if use_alpha_pct:
                expanded_alpha.append(np.full((n_ct, gap_cols), np.nan))

    data = np.concatenate(expanded_blocks, axis=1)

    # ✅ FIX: alpha must be finite and within [0,1]
    if use_alpha_pct:
        alpha_plot = np.concatenate(expanded_alpha, axis=1)

        # nan -> 0 (transparent); inf -> bounds; then clip
        alpha_plot = np.nan_to_num(alpha_plot, nan=0.0, posinf=1.0, neginf=0.0)
        alpha_plot = np.clip(alpha_plot, 0.0, 1.0)

        # also make alpha 0 where data is nan (gap columns / missing values)
        alpha_plot[np.isnan(data)] = 0.0
    else:
        alpha_plot = 1.0

    # figsize
    n_cols = data.shape[1]
    fig_w = max(min_fig_w, col_inch * n_cols + 10)
    fig_w = min(max_fig_w, fig_w)

    fig_h = max(min_fig_h, 6 + row_inch * n_ct)
    fig_h = min(max_fig_h, fig_h)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(
        data,
        aspect="auto",
        interpolation="nearest",
        cmap=cmap2,
        vmin=vmin, vmax=vmax,
        alpha=alpha_plot
    )

    # y ticks
    ax.set_yticks(np.arange(n_ct))
    ax.set_yticklabels(celltype_order, fontsize=11)
    ax.invert_yaxis()

    # x ticks (tissue repeated per gene block; gaps skipped)
    xticks, xticklabels = [], []
    x = 0
    for gi, g in enumerate(gene_order):
        for tj, t in enumerate(tissues):
            xticks.append(x + tj)
            xticklabels.append(str(t))
        x += n_t
        if gi < len(gene_order) - 1:
            x += gap_cols

    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=90, fontsize=9)

    # top axis: gene labels
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    centers = []
    x = 0
    for gi, g in enumerate(gene_order):
        centers.append(x + (n_t - 1) / 2)
        x += n_t + (gap_cols if gi < len(gene_order) - 1 else 0)
    ax2.set_xticks(centers)
    ax2.set_xticklabels(gene_order, fontsize=11, fontweight="bold")
    ax2.tick_params(axis="x", length=0)

    # separators between genes
    x = 0
    for gi in range(1, len(gene_order)):
        x += n_t
        ax.axvline(x - 0.5, color="black", linewidth=1.0, alpha=0.6)
        x += gap_cols

    ax.set_title(f"{group_name} — Heatmap (gene blocks × {n_t} tissues)", fontsize=14, fontweight="bold", pad=18)
    ax.set_ylabel("Cell Type", fontsize=12)
    ax.set_xlabel("")

    cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label("Scaled expression (gene-wise z)", rotation=270, labelpad=14, fontsize=10)

    plt.subplots_adjust(left=0.28, right=0.90, top=0.90, bottom=0.18)

    dpi_used = _safe_savefig(fig, out_pdf, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✓ Saved: {Path(out_pdf).name} (figsize={fig_w:.1f}x{fig_h:.1f} in, dpi={dpi_used})")


# -----------------------------
# Batch export
# -----------------------------
MAX_GENES_PER_FIG = 18
groups_to_plot = [g for g in MECH_PRIORITY if g in df["group"].unique()]
print("Heatmap groups:", len(groups_to_plot))
print("Tissues:", TISSUES_4)

for gname in groups_to_plot:
    sub = df[df["group"] == gname].copy()
    if sub.empty:
        continue

    gene_order = [x for x in MECH_GENE_SETS.get(gname, []) if x in sub["gene"].unique()]
    if len(gene_order) == 0:
        continue

    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(gname))

    if len(gene_order) > MAX_GENES_PER_FIG:
        for part_i, start in enumerate(range(0, len(gene_order), MAX_GENES_PER_FIG), 1):
            genes_part = gene_order[start:start + MAX_GENES_PER_FIG]
            out_pdf = OUTDIR_HM / f"group_{safe}_part{part_i:02d}_HEATMAP_4tissue_gene_blocks.pdf"
            plot_group_heatmap_gene_blocks_4tissue(
                sub, f"{gname} (part {part_i:02d})", out_pdf,
                gene_order=genes_part,
                celltype_order=CELLTYPE_ORDER,
                tissues=TISSUES_4,
                use_alpha_pct=True,
                gap_cols=1
            )
    else:
        out_pdf = OUTDIR_HM / f"group_{safe}_HEATMAP_4tissue_gene_blocks.pdf"
        plot_group_heatmap_gene_blocks_4tissue(
            sub, gname, out_pdf,
            gene_order=gene_order,
            celltype_order=CELLTYPE_ORDER,
            tissues=TISSUES_4,
            use_alpha_pct=False,
            gap_cols=1
        )

print("\nDONE:", OUTDIR_HM)


OUTDIR_HM = /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing/by_group_4tissue_gene_blocks/heatmaps_gene_blocks
Heatmap groups: 15
Tissues: ['nose', 'lung parenchyma', 'sinus', 'respiratory airway']
  ✓ Saved: group_Classic_mechanosensitive_channels_HEATMAP_4tissue_gene_blocks.pdf (figsize=22.0x45.5 in, dpi=300)
  ✓ Saved: group_TMC_MET_complex_HEATMAP_4tissue_gene_blocks.pdf (figsize=30.5x45.5 in, dpi=300)
  ✓ Saved: group_TRP-related_mechanosensors_HEATMAP_4tissue_gene_blocks.pdf (figsize=22.0x45.5 in, dpi=300)
  ✓ Saved: group_K2P_stretch-sensitive_K_channels_HEATMAP_4tissue_gene_blocks.pdf (figsize=22.0x45.5 in, dpi=300)
  ✓ Saved: group_ENaC_ASIC_family_HEATMAP_4tissue_gene_blocks.pdf (figsize=22.0x45.5 in, dpi=300)
  ✓ Saved: group_Integrins_ECM_force_receptors__part01_HEATMAP_4tissue_gene_blocks.pdf (figsize=38.5x45.5 in, dpi=300)
  ✓ Saved: group_Integrins_ECM_force_receptors__part02_HEATMAP_4tissue_gene_blocks.pd

In [44]:
# ================================================================================
# CELL 1/2 — Mechanosensing Heatmap: PREP + Expression Matrix (no plotting)
# ================================================================================

import numpy as np
import pandas as pd
from pathlib import Path
import scanpy as sc
from scipy.sparse import issparse

print("="*80)
print("MECHANOSENSING HEATMAP — CELL 1/2 (PREP)")
print("="*80)

# -----------------------------
# 0) Hard requirements checks
# -----------------------------
_required = ["main", "FIGDIR", "MAIN_L3_KEY", "MECH_GENE_SETS"]
missing = [k for k in _required if k not in globals()]
if missing:
    raise NameError(
        f"Missing required variables: {missing}\n"
        "You must have these defined before running Cell 1:\n"
        "  - main: AnnData\n"
        "  - FIGDIR: pathlib.Path\n"
        "  - MAIN_L3_KEY: obs key (categorical recommended)\n"
        "  - MECH_GENE_SETS: dict(group -> list of genes)"
    )

# Ensure FIGDIR is Path
FIGDIR = Path(FIGDIR)

# Ensure categorical cell type key
if MAIN_L3_KEY not in main.obs.columns:
    raise KeyError(f"MAIN_L3_KEY='{MAIN_L3_KEY}' not found in main.obs columns")
if not pd.api.types.is_categorical_dtype(main.obs[MAIN_L3_KEY]):
    main.obs[MAIN_L3_KEY] = main.obs[MAIN_L3_KEY].astype("category")

# -----------------------------
# 1) Define/patch MECH_PRIORITY, unique_genes, PANEL_1..4
# -----------------------------
if "MECH_PRIORITY" not in globals() or MECH_PRIORITY is None or len(MECH_PRIORITY) == 0:
    # preserve insertion order of dict (py>=3.7)
    MECH_PRIORITY = list(MECH_GENE_SETS.keys())
    print("⚠️  MECH_PRIORITY not found → set to list(MECH_GENE_SETS.keys())")

if "unique_genes" not in globals() or unique_genes is None or len(unique_genes) == 0:
    unique_genes = sorted({g for grp in MECH_PRIORITY for g in MECH_GENE_SETS.get(grp, [])})
    print(f"⚠️  unique_genes not found → built from MECH_GENE_SETS ({len(unique_genes)} genes)")

def _chunk_into_panels(groups, n_panels=4):
    """Fallback: split group list into n roughly equal chunks."""
    groups = list(groups)
    if len(groups) == 0:
        return [[] for _ in range(n_panels)]
    cuts = np.linspace(0, len(groups), n_panels + 1).round().astype(int)
    return [groups[cuts[i]:cuts[i+1]] for i in range(n_panels)]

# PANEL_1..4 fallback if missing
need_panels = any([k not in globals() for k in ["PANEL_1", "PANEL_2", "PANEL_3", "PANEL_4"]])
if need_panels:
    p1, p2, p3, p4 = _chunk_into_panels(MECH_PRIORITY, 4)
    PANEL_1, PANEL_2, PANEL_3, PANEL_4 = p1, p2, p3, p4
    print("⚠️  PANEL_1..4 not found → auto-generated by chunking MECH_PRIORITY into 4 panels")
    print(f"    PANEL_1: {len(PANEL_1)} groups | PANEL_2: {len(PANEL_2)} | PANEL_3: {len(PANEL_3)} | PANEL_4: {len(PANEL_4)}")

# -----------------------------
# 2) Prepare expression matrix (mean per cell type)
# -----------------------------
print("\n[STEP] Preparing expression data for heatmap...")

use_raw = main.raw is not None
gene_universe = main.raw.var_names if use_raw else main.var_names

present_mech_genes = [g for g in unique_genes if g in gene_universe]
if len(present_mech_genes) == 0:
    raise ValueError("No mechanosensing genes found in gene universe (raw.var_names or var_names).")

cell_types = main.obs[MAIN_L3_KEY].cat.categories.tolist()

expr_matrix = []
for ct in cell_types:
    ct_mask = (main.obs[MAIN_L3_KEY] == ct).values
    ct_data = main[ct_mask]

    if use_raw:
        X = ct_data.raw[:, present_mech_genes].X
    else:
        X = ct_data[:, present_mech_genes].X

    if issparse(X):
        means = np.asarray(X.mean(axis=0)).ravel()
    else:
        means = np.asarray(X).mean(axis=0).ravel()

    expr_matrix.append(means)

expr_df = pd.DataFrame(expr_matrix, index=cell_types, columns=present_mech_genes)

# Z-score across cell types per gene
expr_df_scaled = expr_df.apply(lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-10), axis=0)

print(f"  Expression matrix: {expr_df.shape[0]} cell types × {expr_df.shape[1]} genes")
print("  ✓ expr_df / expr_df_scaled ready")

# -----------------------------
# 3) Output directory (no plotting)
# -----------------------------
mech_heatmap_dir = FIGDIR / "mechanosensing_heatmaps"
mech_heatmap_dir.mkdir(parents=True, exist_ok=True)
print(f"\n✓ Output dir ready: {mech_heatmap_dir}")

# (Optional) Save matrices so cell2 fails won't waste computed results
expr_df.to_csv(mech_heatmap_dir / "mech_expr_mean_by_celltype.csv")
expr_df_scaled.to_csv(mech_heatmap_dir / "mech_expr_zscore_by_celltype.csv")
print("✓ Saved CSV: mech_expr_mean_by_celltype.csv, mech_expr_zscore_by_celltype.csv")

print("\n" + "="*80)
print("CELL 1/2 COMPLETE (NO PLOTTING DONE)")
print("="*80)


MECHANOSENSING HEATMAP — CELL 1/2 (PREP)
⚠️  PANEL_1..4 not found → auto-generated by chunking MECH_PRIORITY into 4 panels
    PANEL_1: 4 groups | PANEL_2: 4 | PANEL_3: 3 | PANEL_4: 4

[STEP] Preparing expression data for heatmap...
  Expression matrix: 152 cell types × 136 genes
  ✓ expr_df / expr_df_scaled ready

✓ Output dir ready: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing_heatmaps
✓ Saved CSV: mech_expr_mean_by_celltype.csv, mech_expr_zscore_by_celltype.csv

CELL 1/2 COMPLETE (NO PLOTTING DONE)


In [46]:
# ================================================================================
# CELL 2/2 — Mechanosensing Heatmap: PLOTTING ONLY
# ================================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

print("="*80)
print("MECHANOSENSING HEATMAP — CELL 2/2 (PLOTTING)")
print("="*80)

# -----------------------------
# 1) Colormap (same as dotplot)
# -----------------------------
colors_list = [
    "#1E466E", "#376795", "#528FAD", "#72BCD5", "#AADCE0",
    "#FFE6B7", "#FFD06F", "#F7AA58", "#EF8A47", "#E76254"
]
custom_cmap = LinearSegmentedColormap.from_list("mech_cmap", colors_list, N=256)

# -----------------------------
# 2) Plot function (with optional clustering)
# -----------------------------
def _cluster_order(mat, axis=0, metric="correlation", method="average"):
    """Return reordered indices for clustering rows or cols."""
    arr = np.asarray(mat)
    if axis == 1:
        arr = arr.T
    if arr.shape[0] <= 2:
        return np.arange(arr.shape[0])
    d = pdist(arr, metric=metric)
    Z = linkage(d, method=method)
    return leaves_list(Z)

def plot_mech_heatmap(expr_data, genes_subset, groups_subset, title, filename,
                      figsize=(18, 10), cluster_rows=True, cluster_cols=False):
    # Filter genes
    genes_in_data = [g for g in genes_subset if g in expr_data.columns]
    if len(genes_in_data) == 0:
        print(f"  ⚠️  No genes available for: {title}")
        return

    # Order genes by group (and optionally cluster within each group)
    genes_ordered = []
    group_boundaries = [0]

    for group in groups_subset:
        group_genes = [g for g in MECH_GENE_SETS[group] if g in genes_in_data]
        if len(group_genes) == 0:
            group_boundaries.append(len(genes_ordered))
            continue

        if cluster_cols and len(group_genes) >= 3:
            sub = expr_data[group_genes].values
            col_ord = _cluster_order(sub, axis=1)
            group_genes = [group_genes[i] for i in col_ord]

        genes_ordered.extend(group_genes)
        group_boundaries.append(len(genes_ordered))

    if len(genes_ordered) == 0:
        print(f"  ⚠️  No genes after grouping for: {title}")
        return

    plot_data = expr_data[genes_ordered].copy()

    # Cluster rows if requested
    if cluster_rows and plot_data.shape[0] >= 3:
        row_ord = _cluster_order(plot_data.values, axis=0)
        plot_data = plot_data.iloc[row_ord, :]

    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        plot_data,
        cmap=custom_cmap,
        center=0,
        vmin=-2,
        vmax=2,
        cbar_kws={'label': 'Scaled Expression (Z-score)', 'shrink': 0.8},
        xticklabels=True,
        yticklabels=True,
        linewidths=0,
        ax=ax,
        robust=True
    )

    # Vertical separators between groups
    for boundary in group_boundaries[1:-1]:
        ax.axvline(x=boundary, color='white', linewidth=2, linestyle='-')

    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

    # Group labels on top
    group_centers, group_labels = [], []
    for i, group in enumerate(groups_subset):
        start = group_boundaries[i]
        end = group_boundaries[i + 1]
        if end > start:
            group_centers.append((start + end) / 2)
            group_labels.append(group)

    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(group_centers)
    ax2.set_xticklabels(group_labels, rotation=20, ha='left', fontsize=9, fontweight='bold')
    ax2.tick_params(axis='x', which='both', length=0)

    ax.set_title(title, fontsize=12, fontweight='bold', pad=40)
    ax.set_xlabel('')
    ax.set_ylabel('Cell Type', fontsize=10)

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved: {Path(filename).name}")

# -----------------------------
# 3) 4 Panels
# -----------------------------
print("\n[STEP] Generating 4 mechanosensing heatmaps...")

# Panel 1
print("Panel 1: Primary Sensors")
panel1_genes = [g for grp in PANEL_1 for g in MECH_GENE_SETS.get(grp, []) if g in expr_df_scaled.columns]
plot_mech_heatmap(
    expr_df_scaled, panel1_genes, PANEL_1,
    "Mechanosensing Panel 1",
    mech_heatmap_dir / "mech_heatmap_panel1_primary_sensors.pdf",
    figsize=(40, 40),
    cluster_rows=True,
    cluster_cols=False
)

# Panel 2
print("\nPanel 2: Surface Receptors")
panel2_genes = [g for grp in PANEL_2 for g in MECH_GENE_SETS.get(grp, []) if g in expr_df_scaled.columns]
plot_mech_heatmap(
    expr_df_scaled, panel2_genes, PANEL_2,
    "Mechanosensing Panel 2",
    mech_heatmap_dir / "mech_heatmap_panel2_surface_receptors.pdf",
    figsize=(40, 40),
    cluster_rows=True,
    cluster_cols=False
)

# Panel 3
print("\nPanel 3: Cellular Transduction")
panel3_genes = [g for grp in PANEL_3 for g in MECH_GENE_SETS.get(grp, []) if g in expr_df_scaled.columns]
plot_mech_heatmap(
    expr_df_scaled, panel3_genes, PANEL_3,
    "Mechanosensing Panel 3",
    mech_heatmap_dir / "mech_heatmap_panel3_cellular_transduction.pdf",
    figsize=(40, 40),
    cluster_rows=True,
    cluster_cols=False
)

# Panel 4
print("\nPanel 4: Nuclear & ECM")
panel4_genes = [g for grp in PANEL_4 for g in MECH_GENE_SETS.get(grp, []) if g in expr_df_scaled.columns]
plot_mech_heatmap(
    expr_df_scaled, panel4_genes, PANEL_4,
    "Mechanosensing Panel 4",
    mech_heatmap_dir / "mech_heatmap_panel4_nuclear_ECM.pdf",
    figsize=(40, 40),
    cluster_rows=True,
    cluster_cols=False
)

# -----------------------------
# 4) Overview heatmap (all groups)
# -----------------------------
print("\n[BONUS] Generating combined overview heatmap (all genes)...")

all_genes_ordered = []
all_group_boundaries = [0]
for group in MECH_PRIORITY:
    group_genes = [g for g in MECH_GENE_SETS.get(group, []) if g in expr_df_scaled.columns]
    all_genes_ordered.extend(group_genes)
    all_group_boundaries.append(len(all_genes_ordered))

if len(all_genes_ordered) == 0:
    print("  ⚠️  No genes for overview heatmap. Skip.")
else:
    plot_data_all = expr_df_scaled[all_genes_ordered].copy()

    # optional row clustering
    if plot_data_all.shape[0] >= 3:
        row_ord = _cluster_order(plot_data_all.values, axis=0)
        plot_data_all = plot_data_all.iloc[row_ord, :]

    fig, ax = plt.subplots(figsize=(24, 12))
    sns.heatmap(
        plot_data_all,
        cmap=custom_cmap,
        center=0,
        vmin=-2,
        vmax=2,
        cbar_kws={'label': 'Scaled Expression (Z-score)', 'shrink': 0.6},
        xticklabels=True,
        yticklabels=True,
        linewidths=0,
        ax=ax,
        robust=True
    )

    for boundary in all_group_boundaries[1:-1]:
        ax.axvline(x=boundary, color='white', linewidth=2.5, linestyle='-')

    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='center', fontsize=6)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)

    group_centers, group_labels = [], []
    for i, group in enumerate(MECH_PRIORITY):
        start = all_group_boundaries[i]
        end = all_group_boundaries[i + 1]
        if end > start:
            group_centers.append((start + end) / 2)
            label = group.replace("mechanosensitive", "MS").replace("mechanotransduction", "MT")
            group_labels.append(label)

    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(group_centers)
    ax2.set_xticklabels(group_labels, rotation=25, ha='left', fontsize=8, fontweight='bold')
    ax2.tick_params(axis='x', which='both', length=0)

    ax.set_title("Complete Mechanosensing Gene Expression Overview", fontsize=14, fontweight='bold', pad=50)
    ax.set_xlabel('')
    ax.set_ylabel('Cell Type', fontsize=11)

    plt.tight_layout()
    overview_file = mech_heatmap_dir / "mech_heatmap_overview_all_genes.pdf"
    plt.savefig(overview_file, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved: {overview_file.name}")

print("\n" + "="*80)
print("MECHANOSENSING HEATMAP GENERATION COMPLETE")
print("="*80)
print(f"Output directory: {mech_heatmap_dir}")
print("Generated files:")
for f in sorted(mech_heatmap_dir.glob("mech_heatmap_*.pdf")):
    print(f"  - {f.name}")


MECHANOSENSING HEATMAP — CELL 2/2 (PLOTTING)

[STEP] Generating 4 mechanosensing heatmaps...
Panel 1: Primary Sensors
  ✓ Saved: mech_heatmap_panel1_primary_sensors.pdf

Panel 2: Surface Receptors
  ✓ Saved: mech_heatmap_panel2_surface_receptors.pdf

Panel 3: Cellular Transduction
  ✓ Saved: mech_heatmap_panel3_cellular_transduction.pdf

Panel 4: Nuclear & ECM
  ✓ Saved: mech_heatmap_panel4_nuclear_ECM.pdf

[BONUS] Generating combined overview heatmap (all genes)...
  ✓ Saved: mech_heatmap_overview_all_genes.pdf

MECHANOSENSING HEATMAP GENERATION COMPLETE
Output directory: /home/h2048/data/py/1128/bbknn_annotation_analysis/merge_L3_from_subobjects_20260209_205626/figures/mechanosensing_heatmaps
Generated files:
  - mech_heatmap_overview_all_genes.pdf
  - mech_heatmap_panel1_primary_sensors.pdf
  - mech_heatmap_panel2_surface_receptors.pdf
  - mech_heatmap_panel3_cellular_transduction.pdf
  - mech_heatmap_panel4_nuclear_ECM.pdf


### 9.8 Export Mechanosensing Gene Expression Summary

In [32]:
# Export summary table of mechanosensing gene expression per cell type
summary_stats = []

for group in MECH_PRIORITY:
    group_genes = [g for g in present_mech_genes if gene_to_group[g] == group]
    
    if len(group_genes) == 0:
        continue
    
    for ct in main.obs[MAIN_L3_KEY].cat.categories:
        ct_cells = main[main.obs[MAIN_L3_KEY] == ct]
        
        for gene in group_genes:
            if gene in ct_cells.var_names:
                # Get expression data
                if use_raw:
                    expr_data = ct_cells.raw[:, gene].X
                else:
                    expr_data = ct_cells[:, gene].X
                
                if hasattr(expr_data, 'toarray'):
                    expr_data = expr_data.toarray().flatten()
                else:
                    expr_data = expr_data.flatten()
                
                pct_exp = 100 * (expr_data > 0).sum() / len(expr_data)
                mean_exp = expr_data.mean()
                
                summary_stats.append({
                    'group': group,
                    'gene': gene,
                    'cell_type': ct,
                    'mean_expression': mean_exp,
                    'pct_expressed': pct_exp,
                    'n_cells': len(expr_data)
                })

summary_df = pd.DataFrame(summary_stats)
summary_file = TABDIR / "mechanosensing_expression_summary.csv"
summary_df.to_csv(summary_file, index=False)

print(f"\n[EXPORT] Mechanosensing expression summary: {summary_file}")
print(f"  Total records: {len(summary_df):,}")
print(f"  Genes: {summary_df['gene'].nunique()}")
print(f"  Cell types: {summary_df['cell_type'].nunique()}")
print(f"  Groups: {summary_df['group'].nunique()}")

KeyboardInterrupt: 

## 10. Summary & Output Manifest

In [ ]:
print("\n" + "="*80)
print("PIPELINE COMPLETE")
print("="*80)

print(f"\n[OUTPUT DIRECTORY] {OUTDIR}\n")

print("[FILES GENERATED]")
print(f"\n  Main AnnData:")
print(f"    {UPDATED_H5AD}")
print(f"    - Cells: {main.n_obs:,}")
print(f"    - Genes: {main.n_vars:,}")
print(f"    - L2 categories: {main.obs[MAIN_L2_KEY].nunique()}")
print(f"    - L3 categories: {main.obs[MAIN_L3_KEY].nunique()}")

print(f"\n  Tables:")
for f in sorted(TABDIR.glob("*.csv")):
    print(f"    {f.name}")

print(f"\n  Figures:")
for f in sorted(FIGDIR.glob("*.pdf")):
    print(f"    {f.name}")

print("\n" + "="*80)
print(f"Timestamp: {NOW}")
print("="*80)